# CareFinder: evidence-backed healthcare facility search for India

**CMU DSC Datathon, Fall 2026.** This notebook covers the data processing and the scoring/ranking engine behind CareFinder; the web app (map UI, report form and admin review) is in the GitHub repo.

CareFinder answers *"where can I get service X near me, and why should I trust that answer?"* over the provided 9,989-facility dataset. Every recommendation answers two questions:

* **WHY this facility?** A verbatim quote from the facility's own record shows that it offers the service; a WHO SARA-style readiness score lists which required staff and equipment are documented; a GRADE-style certainty level explains how much to trust that evidence; and a one-line note explains the facility's position in each ranking.
* **HOW did we decide?** An LLM reads every record but may only *cite* item IDs, so quotes are always verbatim. Eligibility, readiness, certainty and distance are each computed with a published method, then combined by reciprocal rank fusion. No weights are hand-tuned.

| Part | Content |
|---|---|
| 0 | Backend source code (written to `nb_src/` and imported below) |
| 1 | Exploration: data quality findings |
| 2 | Evidence extraction: LLM vs rules, adjudication, verbatim-quote check |
| 3 | Scoring and ranking: the WHY and HOW text for real queries |
| 4 | Online query engine: latency |
| 5 | Local need (NFHS-5): suggestions and the impact / policy analysis |
| 6 | Feedback loop: admin corrections applied to the data |

**To run:** clone the repo, open this notebook from the repo root (locally or on Databricks Free Edition), and run all cells. Data is read from `care_backend/data/`. The LLM extraction itself is not re-run here; it needs an API key. Its cached output tables are loaded instead.

In [1]:
%pip install -q pandas numpy scikit-learn pyyaml pyarrow

In [2]:
import os, sys, json, time, random, statistics
from pathlib import Path
import pandas as pd, numpy as np

DATA_DIR = Path(os.environ.get("CARE_DATA_DIR", "care_backend/data"))     # repo layout
SRC = Path("nb_src"); SRC.mkdir(exist_ok=True); sys.path.insert(0, str(SRC))
FULL = next((DATA_DIR / f for f in ["facilities_final_9989.parquet", "facilities_final_9989.csv"]
             if (DATA_DIR / f).exists()), None)
os.environ.update(CARE_DATA_DIR=str(DATA_DIR), DICT_FILE=str(DATA_DIR / "datathon-service-dictionary.yaml"),
                  OUT_DIR=str(DATA_DIR / "llm_out_all"), DATA_FILE=str(DATA_DIR / "facilities_slim.csv"))
print("data dir:", DATA_DIR.resolve()); print("full facility table:", FULL)

data dir: /home/claude/nbrun/care_backend/data
full facility table: care_backend/data/facilities_final_9989.csv


## Part 0: Backend source code
These are the exact modules the deployed app runs. Each cell writes one module to `nb_src/`, and the cells below import them.

* `llm_preprocess.py`: evidence extraction. Rule-based extraction for every facility, an LLM upgrade in priority order, and a merge; quotes are looked up by item ID, and confidence tiers are computed deterministically.
* `adjudicate.py`: second, focused LLM pass for the "rules found it, LLM said none" disagreements.
* `rank_services.py`: eligibility, readiness (WHO SARA), certainty (GRADE / WHO DQR), weighted reciprocal-rank-fusion ranking, and per-service BallTree indexes.
* `suggest.py`: services suggested from local health need (NFHS-5).
* `server.py`: HTTP backend for ranking, the report inbox and admin corrections (with an audit trail); also produces the WHY/HOW text.

In [3]:
%%writefile nb_src/llm_preprocess.py
"""
LLM evidence extraction: cleaned facility table + service dictionary -> evidence tables.

Outputs (in OUT_DIR):
  llm_raw.jsonl        one LLM response per facility (also acts as a resume cache)
  evidence.csv         one row per (facility, slot, cited item): verbatim quote + provenance
  slot_status.csv      one row per (facility, resource): status, tier, conf, best quote
  service_claims.csv   one row per (facility, candidate service): claim type, status, gate

The LLM only cites item IDs. Quotes are looked up in code, so they are always verbatim.
Confidence is computed deterministically from the cited evidence, never by the LLM.
"""
import json, os, re
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
import yaml

# ============================ CONFIG ============================
# Every setting can be overridden with an environment variable; no code edits needed.
HERE = os.path.dirname(os.path.abspath(__file__))

def _find(env, patterns):
    """Use $env if set, else the first file in this script's folder matching a pattern."""
    import glob
    if os.getenv(env):
        return os.getenv(env)
    for p in patterns:
        hits = sorted(glob.glob(os.path.join(HERE, p)))
        if hits:
            return hits[0]
    raise FileNotFoundError(f"No file for {env}: put one matching {patterns} next to this script, "
                            f"or set {env}=/path/to/file")

def _list(env, default):
    v = os.getenv(env)
    return [x.strip() for x in v.split(",") if x.strip()] if v else default

def _int(env, default):
    v = os.getenv(env)
    return default if v in (None, "") else (None if v.lower() == "none" else int(v))

DATA_CSV   = _find("DATA_FILE", ["facilities_final*.csv", "facilities_final*.parquet"])
DICT_YAML  = _find("DICT_FILE", ["*service-dictionary*.yaml", "*service_dictionary*.yaml"])
OUT_DIR    = os.getenv("OUT_DIR", os.path.join(HERE, "llm_out"))
SERVICES   = _list("SERVICES", ["dialysis", "mri", "ct_scan", "c_section", "icu", "x_ray",
                                "emergency_care", "blood_bank", "chemotherapy"])   # SERVICES=all for every service
if SERVICES == ["all"]:
    SERVICES = None
LIMIT      = _int("LIMIT", None)             # e.g. LIMIT=50 for a quick test on the first 50 rows
WORKERS    = _int("WORKERS", 32)             # deepseek-flash allows 2500 concurrent; 32-64 is plenty

# Hybrid mode: rule-based extraction for every facility, LLM upgrades a prioritised subset.
USE_LLM          = os.getenv("USE_LLM", "0") == "1"
LLM_MAX_CALLS    = _int("LLM_MAX_CALLS", None)          # stop after this many LLM calls
LLM_MAX_MINUTES  = _int("LLM_MAX_MINUTES", 30)          # stop submitting new LLM calls after this long
PRIORITY_CITIES  = _list("PRIORITY_CITIES", ["Pune", "Mumbai", "Jaipur"])   # first, then hospitals, then rest

LLM_PROVIDER = os.getenv("LLM_PROVIDER", "openai")     # "openai" (any OpenAI-compatible API) | "anthropic"
LLM_MODEL    = os.getenv("LLM_MODEL", "deepseek-flash")
LLM_API_KEY  = os.getenv("LLM_API_KEY")
LLM_BASE_URL = os.getenv("LLM_BASE_URL", "https://api.deepseek.com")
# DeepSeek V4 models can "think" before answering; extraction does not need it (slower, costlier).
# If the API rejects this field, set LLM_DISABLE_THINKING=0.
LLM_DISABLE_THINKING = os.getenv("LLM_DISABLE_THINKING", "1") == "1"

COL = dict(id="unique_id", name="name", city="address_city", address="address_line1",
           record="source_content_id", specialties="specialties",
           list_fields=["procedure", "equipment", "capability"], text_fields=["description"])
FIELD_PREFIX = {"procedure": "P", "equipment": "E", "capability": "C", "description": "D", "specialties": "S"}

TIER_CONF = {"explicit_count": 1.0, "corroborated": 0.9, "single_text": 0.7, "tag_only": 0.5}
ACCEPTED  = {"explicitly_offered_at_branch", "listed_on_branch_service_menu"}
CONTRADICTING = {"negated", "referral_only"}
# ================================================================


# ---------------- LLM call (the only provider-specific part) ----------------
_CLIENT = None

def _client():
    """One shared, thread-safe client (reuses connections); retries connection errors / 429s."""
    global _CLIENT
    if _CLIENT is None:
        if LLM_PROVIDER == "anthropic":
            import anthropic
            _CLIENT = anthropic.Anthropic(api_key=LLM_API_KEY, max_retries=6, timeout=120)
        else:
            from openai import OpenAI
            _CLIENT = OpenAI(api_key=LLM_API_KEY, base_url=LLM_BASE_URL, max_retries=6, timeout=120)
    return _CLIENT

def call_llm(system: str, user: str) -> str:
    client = _client()
    if LLM_PROVIDER == "anthropic":
        r = client.messages.create(model=LLM_MODEL, max_tokens=2000, temperature=0,
                                   system=system, messages=[{"role": "user", "content": user}])
        return r.content[0].text
    extra = {"thinking": {"type": "disabled"}} if LLM_DISABLE_THINKING and "deepseek" in (LLM_BASE_URL or "") else None
    r = client.chat.completions.create(model=LLM_MODEL, temperature=0, extra_body=extra,
                                       response_format={"type": "json_object"},
                                       messages=[{"role": "system", "content": system},
                                                 {"role": "user", "content": user}])
    return r.choices[0].message.content


# ---------------- text normalisation for the candidate pre-filter ----------------
GENERIC = {"care", "treatment", "service", "services", "surgery", "clinic", "center", "therapy",
           "test", "testing", "unit", "management", "consultation", "procedure", "facility", "any"}

SPELL = [("haemo", "hemo"), ("paed", "ped"), ("oesoph", "esoph"), ("centre", "center"),
         ("anaesth", "anesth"), ("gynaec", "gynec"), ("orthopaed", "orthoped")]

def norm_list(s: str) -> list:
    s = re.sub(r"([a-z])([A-Z])", r"\1 \2", str(s)).lower()
    for a, b in SPELL:
        s = s.replace(a, b)
    return [t[:-1] if len(t) > 3 and t.endswith("s") and not t.endswith("ss") else t
            for t in re.sub(r"[^a-z0-9]+", " ", s).split()]


def norm_tokens(s: str) -> set:
    s = re.sub(r"([a-z])([A-Z])", r"\1 \2", str(s)).lower()
    for a, b in SPELL:
        s = s.replace(a, b)
    toks = re.sub(r"[^a-z0-9]+", " ", s).split()
    return {t[:-1] if len(t) > 3 and t.endswith("s") and not t.endswith("ss") else t for t in toks}


# ---------------- dictionary -> services, slots, pre-filter index ----------------
def load_dictionary(path):
    d = yaml.safe_load(open(path))
    R, S = d["resources"], d["services"]
    if SERVICES:
        S = {k: v for k, v in S.items() if k in SERVICES}

    def expand(slots):  # "a|b" -> a, b ; drop search cues and evidence checks
        out = []
        for s in slots or []:
            for r in s.split("|"):
                if r in R and R[r]["kind"] not in ("search_cue", "evidence_check"):
                    out.append(r)
        return out

    svc = {}
    for sid, s in S.items():
        claim_check = next(e for e in s["essential"] if e != "branch_identity")
        claim_res = R[claim_check]["resource_ref"]
        support = expand(s.get("supporting"))
        for m in (s.get("modality_profiles") or {}).values():
            support += expand(m.get("supporting"))
        for a in s.get("authority_readiness") or []:
            support += expand(a.get("essential")) + expand(a.get("supporting"))
        terms = R[claim_res].get("synonyms", []) + s.get("online_search_terms", []) + [s["display_name"]]
        svc[sid] = dict(name=s["display_name"], claim_resource=claim_res,
                        supporting=sorted(set(support) - {claim_res}),
                        modalities=list((s.get("modality_profiles") or {}).keys()),
                        terms=[t for t in map(norm_tokens, terms) if t - GENERIC],  # skip all-generic terms
                        phrases=[tuple(p) for p in map(norm_list, terms) if set(p) - GENERIC])

    # inverted index: longest token of each term -> [(service, term_tokens)]
    index = defaultdict(list)
    for sid, v in svc.items():
        for t in v["terms"]:
            index[max(t, key=len)].append((sid, t))
    return R, svc, index


# ---------------- facility -> numbered items ----------------
_LIT = re.compile(r"'(?:[^'\\]|\\.)*'|\"(?:[^\"\\]|\\.)*\"")

def as_list(x):
    """Accepts real lists, JSON lists, and NumPy-repr strings like "['a' 'b'\\n 'c']"."""
    import ast
    if isinstance(x, (list, tuple)):
        return list(x)
    if hasattr(x, "tolist"):
        return x.tolist()
    if not isinstance(x, str) or not x.strip():
        return []
    s = x.strip()
    try:
        v = json.loads(s)
        if isinstance(v, list):
            return v
    except Exception:
        pass
    return [ast.literal_eval(m) for m in _LIT.findall(s)]

def build_items(row):
    items = {}
    for f in COL["list_fields"]:
        for i, t in enumerate(dict.fromkeys(str(x).strip() for x in as_list(row[f]) if str(x).strip()), 1):
            items[f"{FIELD_PREFIX[f]}{i}"] = (f, t)
    for f in COL["text_fields"]:
        sents = re.split(r"(?<=[.!?])\s+", row[f]) if isinstance(row[f], str) else []
        for i, t in enumerate((s.strip() for s in sents if s.strip()), 1):
            items[f"{FIELD_PREFIX[f]}{i}"] = (f, t)
    for i, t in enumerate(dict.fromkeys(as_list(row[COL["specialties"]])), 1):
        items[f"S{i}"] = ("specialties", str(t))
    return items

def candidate_services(items, index):
    cands = set()
    for f, text in items.values():
        if f == "specialties":          # a specialty tag alone never establishes a service
            continue
        toks = norm_tokens(text)
        for t in toks:
            for sid, term in index.get(t, ()):
                if term <= toks:
                    cands.add(sid)
    return sorted(cands)


# ---------------- rule-based extractor (fallback; same output format as the LLM) ----------------
NEG      = re.compile(r"\b(no|not|without|unavailable|non[- ]?functional|closed|discontinued)\b", re.I)
REFERRAL = re.compile(r"\brefer(red|ral|s)?\b", re.I)
PLANNED  = re.compile(r"\b(coming soon|planned|upcoming|proposed|under construction|will be)\b", re.I)
COUNT    = re.compile(r"[:\-]\s*(\d{1,4})\b|^(\d{1,4})\s|\b(\d{1,4})\s+(?:\w+\s){0,2}"
                      r"(?:machines?|beds?|units?|ventilators?|scanners?|stations?|tables?|chairs?)\b", re.I)
_res_terms = {}

def resource_terms(R, r):
    if r not in _res_terms:
        terms = [tuple(norm_list(t)) for t in R[r].get("synonyms", []) + [r.replace("_", " ")]]
        _res_terms[r] = [t for t in terms if set(t) - GENERIC]
    return _res_terms[r]

def term_matches(phrase, toks_list, toks_set):
    """Long-token terms: word order ignored. Terms with a short token (HD, RO, CT, ICU): exact phrase."""
    if all(len(t) > 3 for t in phrase):
        return set(phrase) <= toks_set
    n = len(phrase)
    return any(tuple(toks_list[i:i + n]) == phrase for i in range(len(toks_list) - n + 1))

def hits(items, phrases, skip_specialties=True):
    out = []
    for k, (f, text) in items.items():
        if skip_specialties and f == "specialties":
            continue
        tl = norm_list(text); ts = set(tl)
        if any(term_matches(p, tl, ts) for p in phrases):
            out.append(k)
    return out

def rule_extract(items, cands, svc, R):
    out = {"services": [], "resources": []}
    for s in cands:
        ids = hits(items, svc[s]["phrases"])
        texts = [items[k][1] for k in ids]
        fields = {items[k][0] for k in ids}
        if ids and all(REFERRAL.search(t) for t in texts):
            claim = "referral_only"
        elif ids and all(NEG.search(t) for t in texts):
            claim = "negated"
        elif ids and all(PLANNED.search(t) for t in texts):
            claim = "planned_only"
        elif fields - {"equipment"}:
            claim = "listed_on_branch_service_menu"
        elif ids:
            claim = "equipment_only"
        else:
            claim = "mention_without_service_claim"
        keep = [k for k in ids if not NEG.search(items[k][1])] if claim.startswith("listed") else ids
        mod = next((m for m in svc[s]["modalities"] if any(m[:6] in norm_tokens(t) or m[:6] in t.lower() for t in texts)), None)
        out["services"].append({"id": s, "claim": claim, "items": keep, "modality": mod})
    for r in sorted({r for s in cands for r in svc[s]["supporting"]}):
        ids = hits(items, resource_terms(R, r), skip_specialties=R[r]["kind"] != "staff")
        if not ids:
            continue
        neg = [k for k in ids if NEG.search(items[k][1])]
        pos = [k for k in ids if k not in neg]
        count = None
        if R[r]["kind"] in ("equipment", "staff"):
            for k in pos:
                m = COUNT.search(items[k][1])
                if m:
                    count = int(next(g for g in m.groups() if g)); break
        out["resources"].append({"id": r, "confirmed": pos, "contradicted": neg, "count": count})
    return out


# ---------------- prompt ----------------
SYSTEM = """You extract evidence about healthcare services from one facility record.
You are given numbered ITEMS (verbatim text from the record), CANDIDATE SERVICES and RESOURCES.
Cite evidence ONLY by item ID. Never invent IDs. Return JSON only.

For each candidate service, classify the record's claim with exactly one claim type:
- explicitly_offered_at_branch: text states this facility provides it
- listed_on_branch_service_menu: it appears in this facility's list of services/procedures
- negated: text says it is not available
- planned_only: coming soon / planned
- referral_only: patients are referred elsewhere for it
- equipment_only: only equipment is mentioned, no service claim
- specialty_only: only a specialty/department tag, no service claim
- other_branch: offered at a different branch/location
- mention_without_service_claim: mentioned in another sense (news, education, disease info)
- none: not mentioned
If the service has modalities, set "modality" when the text states one, else null.
Judge EVERY candidate service independently, even if the same concept also appears in RESOURCES.
A department, section, unit or facility of this facility counts as explicitly_offered_at_branch
(e.g. "Has an Intensive Care Unit", "CT Scan Section is available", "Emergency department").

For RESOURCES: list only those with evidence. "confirmed" = items showing it is present at this
facility; "contradicted" = items saying it is absent. Specialty items (S*) may confirm staff
resources. "count" = the stated number of units/staff if given, else null.

Output:
{"services":[{"id":"...","claim":"...","items":["P2"],"modality":null}],
 "resources":[{"id":"...","confirmed":["E1"],"contradicted":[],"count":null}]}"""

def build_prompt(row, items, cands, svc, R):
    res = sorted({r for s in cands for r in svc[s]["supporting"]})
    lines = [f"FACILITY: {row[COL['name']]} ({row[COL['city']]})", "", "ITEMS:"]
    lines += [f"{k}: {t}" for k, (f, t) in items.items()]
    lines += ["", "CANDIDATE SERVICES:"]
    for s in cands:
        mod = f" [modalities: {', '.join(svc[s]['modalities'])}]" if svc[s]["modalities"] else ""
        lines.append(f"{s}: {svc[s]['name']}{mod}")
    lines += ["", "RESOURCES:"]
    lines += [f"{r}: {', '.join(R[r].get('synonyms', [])[:4])}" for r in res]
    return "\n".join(lines)


# ---------------- run one facility ----------------
def process(row, R, svc, index, method):
    items = build_items(row)
    cands = candidate_services(items, index)
    if not cands:
        resp = {"services": [], "resources": []}
    elif method == "llm":
        try:
            raw = call_llm(SYSTEM, build_prompt(row, items, cands, svc, R))
            resp = json.loads(raw[raw.find("{"): raw.rfind("}") + 1])
        except Exception as e:            # rate limit / timeout / bad JSON: keep the rule-based result
            print(f"  LLM failed for {row[COL['id']]}: {type(e).__name__}")
            return None
    else:
        resp = rule_extract(items, cands, svc, R)
    return dict(facility_id=row[COL["id"]], method=method, n_candidates=len(cands),
                items={k: list(v) for k, v in items.items()}, response=resp)


# ---------------- deterministic confidence ----------------
def tier_of(cited, count):
    fields = {f for f, _ in cited}
    if not cited:
        return None
    if count is not None:
        return "explicit_count"
    if len(fields) >= 2:
        return "corroborated"
    if fields - {"specialties"}:
        return "single_text"
    return "tag_only"

def best_quote(cited):
    text = [c for c in cited if c[0] != "specialties"] or cited
    return min(text, key=lambda c: len(c[1]))


def build_tables(results, df):
    meta = df.set_index(COL["id"])
    ev, slots, claims = [], [], []
    for r in results:
        fid, items, resp, method = r["facility_id"], r["items"], r["response"], r.get("method", "llm")
        rec = meta.loc[fid, COL["record"]]
        branch_ok = bool(str(meta.loc[fid, COL["name"]]).strip()) and \
            bool(str(meta.loc[fid, COL["address"]]).strip() or str(meta.loc[fid, COL["city"]]).strip())
        look = lambda ids: [tuple(items[i]) for i in ids if i in items]

        for s in resp.get("services", []):
            claim = s.get("claim", "none")
            if claim == "none":
                continue
            cited = look(s.get("items", []))
            status = "confirmed" if claim in ACCEPTED else "contradicted" if claim in CONTRADICTING else "unknown"
            t = tier_of(cited, None) if status == "confirmed" else None
            claims.append(dict(facility_id=fid, service_id=s["id"], claim_type=claim, status=status, method=method,
                               modality=s.get("modality"), branch_identity_ok=branch_ok,
                               gate_pass=branch_ok and status == "confirmed",
                               tier=t, conf=TIER_CONF.get(t, 0.0),
                               best_field=best_quote(cited)[0] if cited else None,
                               best_quote=best_quote(cited)[1] if cited else None, record_id=rec))
            ev += [dict(facility_id=fid, slot=f"{s['id']}_offered", status=status, claim_type=claim,
                        field=f, quote=q, record_id=rec, count=None, method=method) for f, q in cited]

        for x in resp.get("resources", []):
            conf_items, contra_items = look(x.get("confirmed", [])), look(x.get("contradicted", []))
            count = x.get("count")
            if conf_items and contra_items:
                status = "conflicting"
            elif conf_items:
                status = "confirmed"
            elif contra_items:
                status = "contradicted"
            else:
                continue
            t = tier_of(conf_items, count) if status == "confirmed" else None
            bq = best_quote(conf_items or contra_items)
            slots.append(dict(facility_id=fid, resource=x["id"], status=status, tier=t, method=method,
                              conf=TIER_CONF.get(t, 0.0), count=count,
                              n_fields=len({f for f, _ in conf_items}),
                              best_field=bq[0], best_quote=bq[1], record_id=rec))
            ev += [dict(facility_id=fid, slot=x["id"], status="confirmed", claim_type=None, field=f,
                        quote=q, record_id=rec, count=count, method=method) for f, q in conf_items]
            ev += [dict(facility_id=fid, slot=x["id"], status="contradicted", claim_type=None, field=f,
                        quote=q, record_id=rec, count=None, method=method) for f, q in contra_items]
    return pd.DataFrame(ev), pd.DataFrame(slots), pd.DataFrame(claims)


# ---------------- main ----------------
def main():
    import time
    if USE_LLM and not LLM_API_KEY:
        raise SystemExit("USE_LLM=1 but LLM_API_KEY is not set")
    print(f"data: {DATA_CSV}\ndictionary: {DICT_YAML}\noutput: {OUT_DIR}\n"
          f"services: {SERVICES or 'all'}\nLLM: {'on, ' + LLM_MODEL if USE_LLM else 'off (rule-based only)'}")
    os.makedirs(OUT_DIR, exist_ok=True)
    R, svc, index = load_dictionary(DICT_YAML)
    df = (pd.read_parquet(DATA_CSV) if DATA_CSV.endswith(".parquet")
          else pd.read_csv(DATA_CSV, low_memory=False)).drop_duplicates(COL["id"])
    if LIMIT:
        df = df.head(LIMIT)

    # 1) rule-based pass for every facility (fast, no API)
    results = {}
    for _, row in df.iterrows():
        r = process(row, R, svc, index, "rule")
        results[r["facility_id"]] = r
    print(f"rule-based: {len(results)} facilities")

    # 2) LLM pass in priority order, resumable, within budget
    cache_path = os.path.join(OUT_DIR, "llm_raw.jsonl")
    llm_done = {}
    if os.path.exists(cache_path):
        for line in open(cache_path):
            r = json.loads(line)
            llm_done[r["facility_id"]] = r
    if USE_LLM:
        need = df[df[COL["id"]].map(lambda f: results[f]["n_candidates"] > 0 and f not in llm_done)]
        prio = (~need[COL["city"]].isin(PRIORITY_CITIES)).astype(int) * 2 + \
               (need.get("facilityTypeId", pd.Series("", index=need.index)) != "hospital").astype(int)
        todo = [row for _, row in need.assign(_p=prio).sort_values("_p", kind="stable").iterrows()]
        if LLM_MAX_CALLS:
            todo = todo[:LLM_MAX_CALLS]
        print(f"LLM: {len(llm_done)} cached, {len(todo)} queued")
        deadline = time.time() + 60 * LLM_MAX_MINUTES
        with open(cache_path, "a") as fout, ThreadPoolExecutor(WORKERS) as pool:
            for i in range(0, len(todo), WORKERS * 4):
                if time.time() > deadline:
                    print("LLM time budget reached"); break
                for r in pool.map(lambda row: process(row, R, svc, index, "llm"), todo[i:i + WORKERS * 4]):
                    if r is None:
                        continue                  # not cached -> retried on the next run
                    fout.write(json.dumps(r) + "\n"); fout.flush()
                    llm_done[r["facility_id"]] = r
                print(f"  LLM {min(i + WORKERS * 4, len(todo))}/{len(todo)}")

    # 3) merge: LLM result replaces the rule-based one where available
    for fid, r in llm_done.items():
        if fid in results:
            results[fid] = r
    n_llm = sum(r["method"] == "llm" for r in results.values())
    print(f"merged: {n_llm} LLM, {len(results) - n_llm} rule-based")

    ev, slots, claims = build_tables(list(results.values()), df)
    ev.to_csv(os.path.join(OUT_DIR, "evidence.csv"), index=False)
    slots.to_csv(os.path.join(OUT_DIR, "slot_status.csv"), index=False)
    claims.to_csv(os.path.join(OUT_DIR, "service_claims.csv"), index=False)
    print(f"evidence={len(ev)}  slot_status={len(slots)}  service_claims={len(claims)}")


if __name__ == "__main__":
    main()


Writing nb_src/llm_preprocess.py


In [4]:
%%writefile nb_src/adjudicate.py
"""
Second-pass check for likely LLM misses.

Targets (facility, service) pairs where the rule-based matcher found a service claim but the
first LLM pass said "none". Each pair is re-asked as ONE focused question: one facility, one
service, only the matched lines. The answer replaces the "none" in llm_raw.jsonl, then the
tables are rebuilt from the cache. Resumable: pairs already re-checked are skipped.

Usage (same environment as the main run):
  SERVICES=all OUT_DIR=./llm_out_all python adjudicate.py
"""
import json, os, shutil
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

import llm_preprocess as L

ACCEPTED = L.ACCEPTED
ALLOWED = ACCEPTED | {"negated", "planned_only", "referral_only", "equipment_only", "specialty_only",
                      "other_branch", "mention_without_service_claim", "none"}

ADJ_SYSTEM = """You check ONE healthcare service against a few verbatim ITEMS from one facility record.
Decide whether THIS facility offers the service. Cite evidence by item ID only. Return JSON only:
{"claim": "<type>", "items": ["C3"]}
Claim types:
- explicitly_offered_at_branch: text says this facility provides it, or it has a department/unit/section for it
- listed_on_branch_service_menu: it appears in this facility's list of services or procedures
- negated / planned_only / referral_only: not available / coming soon / patients referred elsewhere
- equipment_only: only equipment is mentioned, no service claim
- specialty_only: only a specialty or department tag of a different scope
- other_branch: offered at a different branch or location
- mention_without_service_claim: mentioned in another sense (company name, study, nearby place,
  a different service that shares words, e.g. "Pediatrics" and "General Surgery" listed separately
  is NOT pediatric surgery)
- none: not mentioned"""


def build_prompt(name, city, sid, svc, R, items, ids):
    syn = R[svc[sid]["claim_resource"]].get("synonyms", [])[:5]
    lines = [f"FACILITY: {name} ({city})",
             f"SERVICE: {svc[sid]['name']}" + (f" (also called: {', '.join(syn)})" if syn else ""),
             "", "ITEMS:"]
    lines += [f"{i}: {items[i][1]}" for i in ids]
    return "\n".join(lines)


def main():
    R, svc, index = L.load_dictionary(L.DICT_YAML)
    cache_path = os.path.join(L.OUT_DIR, "llm_raw.jsonl")
    records = [json.loads(line) for line in open(cache_path)]
    df = (pd.read_parquet(L.DATA_CSV) if L.DATA_CSV.endswith(".parquet")
          else pd.read_csv(L.DATA_CSV, low_memory=False)).drop_duplicates(L.COL["id"]).set_index(L.COL["id"])

    # 1) find pairs: rules say offered, LLM said none (or skipped the service), not yet re-checked
    tasks = []
    for ri, rec in enumerate(records):
        if rec.get("method") != "llm" or not rec["items"]:
            continue
        items = {k: tuple(v) for k, v in rec["items"].items()}
        given = {s["id"]: s for s in rec["response"].get("services", [])}
        for sid in L.candidate_services(items, index):
            prev = given.get(sid)
            if prev is not None and (prev.get("claim") != "none" or prev.get("adjudicated")):
                continue
            rule = L.rule_extract(items, [sid], svc, R)["services"][0]
            if rule["claim"] not in ACCEPTED or not rule["items"]:
                continue
            tasks.append((ri, sid, rule["items"][:8]))
    print(f"pairs to re-check: {len(tasks)}")

    # 2) focused LLM calls
    def ask(t):
        ri, sid, ids = t
        rec = records[ri]
        items = {k: tuple(v) for k, v in rec["items"].items()}
        fac = df.loc[rec["facility_id"]]
        try:
            raw = L.call_llm(ADJ_SYSTEM, build_prompt(fac[L.COL["name"]], fac[L.COL["city"]], sid, svc, R, items, ids))
            ans = json.loads(raw[raw.find("{"): raw.rfind("}") + 1])
        except Exception as e:
            print(f"  failed {rec['facility_id']} / {sid}: {type(e).__name__}")
            return None
        claim = ans.get("claim") if ans.get("claim") in ALLOWED else "none"
        cited = [i for i in ans.get("items", []) if i in ids]
        if claim in ACCEPTED and not cited:
            cited = ids[:1]                       # accepted but no citation: cite the matched line
        return ri, sid, claim, cited

    with ThreadPoolExecutor(L.WORKERS) as pool:
        results = [r for r in pool.map(ask, tasks) if r]

    # 3) write answers into the cache (backup first)
    flips = {}
    for ri, sid, claim, cited in results:
        services = records[ri]["response"].setdefault("services", [])
        entry = next((s for s in services if s["id"] == sid), None)
        if entry is None:
            entry = {"id": sid, "modality": None}
            services.append(entry)
        entry.update(claim=claim, items=cited, adjudicated=True)
        flips[claim] = flips.get(claim, 0) + 1
    backup = cache_path.replace(".jsonl", ".before_adjudication.jsonl")
    if not os.path.exists(backup):
        shutil.copy(cache_path, backup)
    with open(cache_path, "w") as f:
        for rec in records:
            f.write(json.dumps(rec) + "\n")
    print(f"re-checked {len(results)} / {len(tasks)} pairs; verdicts: "
          f"{dict(sorted(flips.items(), key=lambda kv: -kv[1]))}")
    print(f"now offered: {sum(v for k, v in flips.items() if k in ACCEPTED)}")

    # 4) rebuild tables from the updated cache (no new LLM calls)
    L.USE_LLM = False
    L.main()


if __name__ == "__main__":
    main()


Writing nb_src/adjudicate.py


In [5]:
%%writefile nb_src/rank_services.py
"""
Query-time ranking over the outputs of llm_preprocess.py.

Each step follows a published method (no hand-tuned weights):
  1. Hard filter : all essential slots confirmed (dictionary essentials; all-or-nothing as in
                   WHO EmONC signal-function classification) + within radius
  2. Readiness   : WHO SARA service-specific readiness: items scored present/absent, unweighted
                   mean within domain, then across domains. Domains from the resource kind.
  3. Certainty   : GRADE-style four levels (high/moderate/low/very low). Start from the evidence
                   type, rate down one level per concern; concerns follow WHO Data Quality Review
                   dimensions (internal consistency, timeliness). Overall = lowest across essential
                   slots. Kept separate from readiness.
  4. Counts      : SARA scores presence. A count only matters where a standard gives a minimum:
                   item = min(1, n / minimum) for resources listed in UNIT_MIN.
  Ranks          : distance_rank, relevance_rank (readiness, then certainty),
                   fused_rank = reciprocal rank fusion (Cormack, Clarke & Buttcher, SIGIR 2009),
                   k = 60; optional weight w (0.5 = standard RRF ordering).

Performance:
  Startup (once): everything that does not depend on the user's location is precomputed per
                  service: readiness, certainty, authority checks, and a BallTree (haversine)
                  over that service's candidate facilities. Evidence and facilities are hashed.
  Query         : O(1) service lookup + O(log n + k) radius search + O(k log k) ranking,
                  where k = facilities inside the radius. Explanations built for the top results only.

Usage:
  python rank_services.py --service dialysis --lat 18.5204 --lon 73.8567 --radius 15 --w 0.5
"""
import argparse, json, time
from collections import defaultdict

import numpy as np
import pandas as pd
import yaml
from sklearn.neighbors import BallTree

# ============================ CONFIG ============================
import os, glob
HERE = os.path.dirname(os.path.abspath(__file__))

def _find(env, patterns):
    if os.getenv(env):
        return os.getenv(env)
    for p in patterns:
        hits = sorted(glob.glob(os.path.join(HERE, p)))
        if hits:
            return hits[0]
    raise FileNotFoundError(f"No file for {env}: put one matching {patterns} next to this script, or set {env}")

FACILITIES_CSV = _find("DATA_FILE", ["facilities_final*.csv", "facilities_final*.parquet"])
DICT_YAML      = _find("DICT_FILE", ["*service-dictionary*.yaml", "*service_dictionary*.yaml"])
LLM_DIR        = os.getenv("OUT_DIR", os.path.join(HERE, "llm_out"))

COL = dict(id="unique_id", name="name", lat="latitude", lon="longitude",
           addr=["address_line1", "address_line2", "address_city", "address_stateOrRegion"],
           city="address_city", state="address_stateOrRegion",
           updated=["recency_of_page_update", "post_metrics_most_recent_social_media_post_date"],
           urls=["officialWebsite", "facebookLink"])

EARTH_KM     = 6371.0
UPSTREAM_GEO_FLAGS = {"remaining_issue_flags": ["coordinate_pin_distance_over_100km"],
                      "postal_coord_issue_note": ["facility_coord_far_from_postal_gt100km"]}
RRF_K        = 60       # Cormack et al. 2009 default
MIN_RESULTS  = 5        # if fewer confirmed facilities, add "unverified" claims as a fallback tier
MAX_EVIDENCE = 4
STALE_BEFORE = "2025-01-01"
UNIT_MIN     = {}       # resource -> minimum from a published standard, e.g. {"dialysis_machine": N}
                        # only fill with a number you can cite; otherwise presence-only (SARA)
LEVELS       = ["very low", "low", "moderate", "high"]
START_LEVEL  = {"admin_verified": 3, "explicit_count": 3, "corroborated": 3, "single_text": 2, "tag_only": 1}
DOMAIN_OF_KIND = {"staff": "staff", "equipment": "equipment", "infrastructure": "equipment",
                  "service": "diagnostics_and_support"}
CLAIM_TEXT   = {"admin_verified": "confirmed by an administrator after a user report",
                "explicitly_offered_at_branch": "explicitly offered at this facility",
                "listed_on_branch_service_menu": "listed in this facility's services"}
TIER_TEXT    = {"admin_verified": "verified by an administrator", "explicit_count": "with a stated count", "corroborated": "in multiple fields",
                "single_text": "in one field", "tag_only": "only as a specialty tag"}
UNVERIFIED   = {"equipment_only", "specialty_only", "mention_without_service_claim"}
# ================================================================


def haversine(lat1, lon1, lat2, lon2):
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp, dl = p2 - p1, np.radians(lon2 - lon1)
    return 2 * EARTH_KM * np.arcsin(np.sqrt(np.sin(dp / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2))


class ServiceIndex:
    """Precomputed, location-independent data for one service + spatial index over its candidates."""
    def __init__(self, rows):
        self.rows = rows
        coords = np.radians([[r["lat"], r["lon"]] for r in rows])
        self.tree = BallTree(coords, metric="haversine")
        self.tier = np.array([0 if r["capability_tier"] == "confirmed" else 1 for r in rows])
        self.rd = np.array([r["readiness"] if r["assessed"] else 0.0 for r in rows])
        self.lv = np.array([r["level"] for r in rows])
        self.geo = np.array([r["geo_flag"] for r in rows], dtype=int)


class Ranker:
    def __init__(self, corrections=()):
        t0 = time.time()
        d = yaml.safe_load(open(DICT_YAML))
        self.R, self.S = d["resources"], d["services"]
        claims = pd.read_csv(f"{LLM_DIR}/service_claims.csv")
        slots = pd.read_csv(f"{LLM_DIR}/slot_status.csv")
        ev = pd.read_csv(f"{LLM_DIR}/evidence.csv")

        self.fac = self._load_facilities()                                       # facility_id -> dict
        self.slot_idx = {(r.facility_id, r.resource): r for r in slots.itertuples(index=False)}
        self.ev_idx = defaultdict(list)                                          # (facility, slot) -> rows
        for r in ev[ev.status == "confirmed"].itertuples(index=False):
            self.ev_idx[(r.facility_id, r.slot)].append(r)
        self._profiles = {}
        claims = self._apply_corrections(claims, corrections)
        self.claims = claims
        self.index = self._build_service_indexes(claims)                         # service_id -> ServiceIndex
        self.build_seconds = round(time.time() - t0, 2)

    # ---------- facility table + quality flags (hashed by id) ----------
    def _load_facilities(self):
        f = (pd.read_parquet(FACILITIES_CSV) if FACILITIES_CSV.endswith(".parquet")
             else pd.read_csv(FACILITIES_CSV, low_memory=False)).drop_duplicates(COL["id"]).set_index(COL["id"])
        f["lat"] = pd.to_numeric(f[COL["lat"]], errors="coerce")
        f["lon"] = pd.to_numeric(f[COL["lon"]], errors="coerce")
        st = f.groupby(COL["state"])[["lat", "lon"]].transform("median")
        ct = f.groupby(COL["city"])[["lat", "lon"]].transform("median")
        f["geo_flag"] = (haversine(f.lat, f.lon, st.lat, st.lon) > 300) | (haversine(f.lat, f.lon, ct.lat, ct.lon) > 50)
        # upstream location flags (cleaning team): coordinates far from the facility's postcode
        for col, bad in UPSTREAM_GEO_FLAGS.items():
            if col in f.columns:
                f["geo_flag"] |= f[col].astype(str).str.contains("|".join(bad), na=False)
        dates = pd.concat([pd.to_datetime(f[c], errors="coerce", utc=True) for c in COL["updated"]], axis=1)
        f["last_updated"] = dates.max(axis=1)
        f["stale"] = f.last_updated < pd.Timestamp(STALE_BEFORE, tz="UTC")
        f = f[f.lat.notna() & f.lon.notna()]
        f["location_note"] = None
        keep = ["lat", "lon", "geo_flag", "last_updated", "stale", "location_note", COL["name"]] + \
               [c for c in COL["addr"] + COL["urls"] if c in f.columns]
        return f[keep].to_dict("index")

    # ---------- admin corrections (explicit, audited edits applied over the base tables) ----------
    def _apply_corrections(self, claims, corrections):
        from types import SimpleNamespace
        for c in corrections:
            fid, ch, rid = c["facility_id"], c["change"], c["report_id"]
            kind = ch.get("kind")
            if fid not in self.fac and kind != "none":
                continue
            if kind == "set_location":
                self.fac[fid].update(lat=float(ch["latitude"]), lon=float(ch["longitude"]), geo_flag=False,
                                     location_note=f"Coordinates corrected by an administrator (report {rid})")
            elif kind == "set_details":
                for k, v in ch.get("fields", {}).items():
                    self.fac[fid][k] = v
            elif kind == "edit":
                claims = self._apply_edit(claims, fid, c["service_id"], ch, rid)
            elif kind == "set_service":
                sid = c["service_id"]
                claims = claims[~((claims.facility_id == fid) & (claims.service_id == sid))]
                if ch.get("offered"):
                    note = ch.get("note", "").strip() or "Verified by an administrator."
                    row = dict(facility_id=fid, service_id=sid, claim_type="admin_verified", status="confirmed",
                               method="admin", modality=None, branch_identity_ok=True, gate_pass=True,
                               tier="admin_verified", conf=1.0, best_field="admin_correction",
                               best_quote=note, record_id=rid)
                    claims = pd.concat([claims, pd.DataFrame([row])], ignore_index=True)
                    self.ev_idx[(fid, f"{sid}_offered")].insert(
                        0, SimpleNamespace(field="admin_correction", quote=note, record_id=rid))
        return claims

    def _apply_edit(self, claims, fid, sid, ch, rid):
        """Generic admin edit: a list of explicit operations on one facility."""
        from types import SimpleNamespace
        note = (ch.get("note") or "").strip() or "Verified by an administrator."
        for op in ch.get("ops", []):
            kind = op.get("op")
            if kind == "details":
                self.fac[fid].update(op["fields"])
            elif kind == "location":
                self.fac[fid].update(lat=float(op["latitude"]), lon=float(op["longitude"]), geo_flag=False,
                                     location_note=f"Coordinates corrected by an administrator (report {rid})")
            elif kind == "service":
                claims = self._apply_corrections(claims, [dict(facility_id=fid, service_id=sid, report_id=rid,
                    change={"kind": "set_service", "offered": op["offered"], "note": note})])
            elif kind == "remove_quote":
                key = (fid, op["slot"])
                self.ev_idx[key] = [e for e in self.ev_idx.get(key, []) if e.quote != op["quote"]]
                row = self.slot_idx.get((fid, op["slot"]))
                if row is not None and not self.ev_idx[key] and getattr(row, "tier", None) != "admin_verified":
                    self.slot_idx.pop((fid, op["slot"]), None)          # no evidence left -> unknown
            elif kind == "resource":
                r, status = op["resource"], op["status"]
                if status == "unknown":
                    self.slot_idx.pop((fid, r), None)
                    continue
                self.slot_idx[(fid, r)] = SimpleNamespace(
                    facility_id=fid, resource=r, status="confirmed" if status == "present" else "contradicted",
                    tier="admin_verified", conf=1.0, count=op.get("count"), n_fields=1,
                    best_field="admin_correction", best_quote=note, record_id=rid)
                if status == "present":
                    self.ev_idx[(fid, r)].insert(0, SimpleNamespace(field="admin_correction", quote=note, record_id=rid))
        return claims

    # ---------- step 3: certainty (GRADE-style levels, DQR rate-down reasons) ----------
    @staticmethod
    def certainty(tier, fac):
        if tier not in START_LEVEL:
            return 0, ["no positive evidence"]
        if tier == "admin_verified":
            return 3, [TIER_TEXT[tier]]
        level, why, concerns = START_LEVEL[tier], [f"evidence {TIER_TEXT[tier]}"], []
        if fac["geo_flag"]:
            concerns.append("internal inconsistency: coordinates vs stated city/state")
        if fac["stale"]:
            concerns.append(f"timeliness: last updated {fac['last_updated'].date()}")
        return max(0, level - len(concerns)), why + [f"rated down: {c}" for c in concerns]

    # ---------- step 2: profile from the dictionary (cached per service x modality) ----------
    def profile(self, sid, modality):
        key = (sid, modality)
        if key in self._profiles:
            return self._profiles[key]
        s = self.S[sid]

        def usable(slots):
            out = []
            for slot in slots or []:
                alts = [r for r in slot.split("|") if r in self.R and self.R[r]["kind"] in DOMAIN_OF_KIND]
                if alts:
                    out.append(alts)
            return out

        slots = usable(s.get("supporting"))
        mp = (s.get("modality_profiles") or {}).get(modality)
        if mp:
            slots += [a for a in usable(mp.get("supporting")) if a not in slots]
        auth = []
        for a in s.get("authority_readiness") or []:
            if s.get("modality_profiles") and not mp:
                continue                    # modality-specific authority: only when modality is stated
            auth.append(dict(ref=a["authority_ref"], essential=usable(a.get("essential"))))
        self._profiles[key] = (slots, auth)
        return slots, auth

    # ---------- steps 2 + 4: SARA item score ----------
    def slot_score(self, fid, alts):
        best = (0.0, None, "unknown", None)        # score, resource, status, row
        for r in alts:
            row = self.slot_idx.get((fid, r))
            if row is None:
                continue
            if row.status != "confirmed":
                if best[2] == "unknown":
                    best = (0.0, r, row.status, row)
                continue
            score = min(1.0, row.count / UNIT_MIN[r]) if r in UNIT_MIN and pd.notna(row.count) else 1.0
            if score > best[0] or best[2] != "confirmed":
                best = (score, r, "confirmed", row)
        return best

    def readiness(self, fid, slots):
        if not slots:
            return None, {}, []
        domains, detail = defaultdict(list), []
        for alts in slots:
            score, r, status, row = self.slot_score(fid, alts)
            domains[DOMAIN_OF_KIND[self.R[alts[0]]["kind"]]].append(score)
            detail.append(dict(slot="|".join(alts), resource=r, status=status,
                               tier=getattr(row, "tier", None) if status == "confirmed" else None,
                               count=getattr(row, "count", None)))
        dom_means = {k: float(np.mean(v)) for k, v in domains.items()}
        return float(np.mean(list(dom_means.values()))), dom_means, detail

    # ---------- startup: precompute everything location-independent, per service ----------
    def _build_service_indexes(self, claims):
        claims = claims[claims.facility_id.isin(self.fac.keys())]
        keep = claims[(claims.gate_pass == True) |
                      (claims.claim_type.isin(UNVERIFIED) & (claims.branch_identity_ok == True))]
        index = {}
        for sid, grp in keep.groupby("service_id"):
            if sid not in self.S:
                continue
            rows = []
            for c in grp.itertuples(index=False):
                fac = self.fac[c.facility_id]
                slots, auth = self.profile(sid, c.modality if isinstance(c.modality, str) else None)
                rd, dom, detail = self.readiness(c.facility_id, slots)
                auth_res = []
                for a in auth:
                    miss = [" or ".join(self.name_of(r) for r in alts) for alts in a["essential"]
                            if self.slot_score(c.facility_id, alts)[2] != "confirmed"]
                    auth_res.append(dict(ref=a["ref"], met=len(a["essential"]) - len(miss),
                                         total=len(a["essential"]), missing=miss))
                level, why = self.certainty(c.tier if isinstance(c.tier, str) else None, fac)
                rows.append(dict(claim=c, facility_id=c.facility_id, lat=fac["lat"], lon=fac["lon"],
                                 geo_flag=bool(fac["geo_flag"]),
                                 capability_tier="confirmed" if c.gate_pass == True else "unverified",
                                 readiness=rd, domains=dom, detail=detail, auth=auth_res,
                                 level=level, why=why, assessed=rd is not None))
            index[sid] = ServiceIndex(rows)
        return index

    # ---------- explanation (built for returned results only) ----------
    def name_of(self, r):
        return (self.R[r].get("synonyms") or [r])[0].lower()

    def explain(self, sid, row, fac):
        c = row["claim"]
        parts = [f"{self.S[sid]['display_name']} is {CLAIM_TEXT.get(c.claim_type, c.claim_type.replace('_', ' '))}"
                 f" ({TIER_TEXT.get(c.tier, 'unverified')})."]
        conf = [self.name_of(d["resource"]) + (f" ({int(d['count'])})" if pd.notna(d["count"]) and d["count"] else "")
                for d in row["detail"] if d["status"] == "confirmed"]
        if conf:
            parts.append("Supporting evidence: " + ", ".join(conf) + ".")
        unc = []
        if not row["assessed"]:
            unc.append("Readiness not assessed: the service profile has no staff/equipment requirements.")
        for d in row["detail"]:
            label = " or ".join(self.name_of(r) for r in d["slot"].split("|"))
            if d["status"] == "unknown":
                unc.append(f"No evidence of {label}.")
            elif d["status"] in ("contradicted", "conflicting"):
                unc.append(f"Record {d['status']} on {label}.")
            elif d["tier"] == "tag_only":
                unc.append(f"{label.capitalize()} appears only as a specialty tag.")
        for a in row["auth"]:
            if a["missing"]:
                unc.append(f"{a['ref']} checklist items not documented: {', '.join(a['missing'])}.")
        if fac["geo_flag"]:
            unc.append("Coordinates are inconsistent with the stated city/state; location may be inaccurate.")
        if pd.isna(fac["last_updated"]):
            unc.append("No update date available for this record.")
        elif fac["stale"]:
            unc.append(f"Record last updated {fac['last_updated'].date()}; information may be outdated.")
        return " ".join(parts), unc

    def evidence_for(self, fid, sid, detail):
        out, seen = [], set()
        keys = [(fid, f"{sid}_offered")] + [(fid, d["resource"]) for d in detail if d["status"] == "confirmed"]
        for k in keys:
            for r in self.ev_idx.get(k, [])[:1]:
                if (r.field, r.quote) not in seen:
                    seen.add((r.field, r.quote))
                    out.append(dict(field=r.field, quote=r.quote, record_id=r.record_id))
        return out[:MAX_EVIDENCE]

    # ---------- query ----------
    def rank(self, sid, lat, lon, radius_km, w=0.5, top=10):
        empty = dict(service_id=sid, radius_km=radius_km, distance_method="straight_line", results=[])
        idx = self.index.get(sid)                                       # O(1)
        if idx is None:
            return empty

        ids, dist = idx.tree.query_radius(np.radians([[lat, lon]]), r=radius_km / EARTH_KM,
                                          return_distance=True)        # O(log n + k)
        ids, d = ids[0], dist[0] * EARTH_KM
        if len(ids) == 0:
            return empty

        # step 1: hard filter; unverified tier only as a fallback
        tier = idx.tier[ids]
        if (tier == 0).sum() >= MIN_RESULTS:
            m = tier == 0
            ids, d, tier = ids[m], d[m], tier[m]
        rd, lv, geo = idx.rd[ids], idx.lv[ids], idx.geo[ids]
        n = len(ids)

        # distance rank: unreliable coordinates go last
        order = np.lexsort((d, geo))
        dist_rank = np.empty(n, int); dist_rank[order] = np.arange(1, n + 1)
        # relevance rank: tier > readiness (SARA) > certainty (GRADE) > distance
        order = np.lexsort((d, -lv, -rd, tier))
        rel_rank = np.empty(n, int); rel_rank[order] = np.arange(1, n + 1)
        # fused: weighted RRF; confirmed always before unverified
        rrf = w / (RRF_K + rel_rank) + (1 - w) / (RRF_K + dist_rank)
        order = np.lexsort((rel_rank, -rrf, tier))

        results = []
        for pos, j in enumerate(order[:top], 1):
            row = idx.rows[ids[j]]
            fac = self.fac[row["facility_id"]]
            reason, unc = self.explain(sid, row, fac)
            results.append(dict(
                facility_id=row["facility_id"], facility_name=fac[COL["name"]],
                latitude=round(float(fac["lat"]), 5), longitude=round(float(fac["lon"]), 5),
                address=", ".join(str(fac[k]) for k in COL["addr"] if isinstance(fac.get(k), str)),
                location_basis=fac.get("location_note") or
                               ("Facility coordinates from the dataset" + (" (flagged)" if fac["geo_flag"] else "")),
                distance_km=round(float(d[j]), 2),
                distance_rank=int(dist_rank[j]), relevance_rank=int(rel_rank[j]), fused_rank=pos,
                capability_tier=row["capability_tier"],
                confidence=LEVELS[row["level"]], confidence_basis=row["why"],
                readiness=None if row["readiness"] is None else round(row["readiness"], 3),
                readiness_domains={k: round(v, 3) for k, v in row["domains"].items()},
                authority_checks=row["auth"],
                items_documented=[" or ".join(self.name_of(r) for r in d["slot"].split("|")) if d["status"] != "confirmed"
                                  else self.name_of(d["resource"]) for d in row["detail"] if d["status"] == "confirmed"],
                items_missing=[" or ".join(self.name_of(r) for r in d["slot"].split("|"))
                               for d in row["detail"] if d["status"] != "confirmed"],
                relevance_reason=reason,
                evidence=self.evidence_for(row["facility_id"], sid, row["detail"]),
                record_source_urls=[fac[k] for k in COL["urls"] if isinstance(fac.get(k), str)],
                uncertainties=unc))
        return dict(service_id=sid, radius_km=radius_km, distance_method="straight_line",
                    ranking=dict(fused="weighted_rrf", rrf_k=RRF_K, w_relevance=w),
                    n_candidates=n, results=results)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--service", required=True)
    ap.add_argument("--lat", type=float, required=True)
    ap.add_argument("--lon", type=float, required=True)
    ap.add_argument("--radius", type=float, default=30)
    ap.add_argument("--w", type=float, default=0.5, help="weight on relevance in RRF (0..1)")
    ap.add_argument("--top", type=int, default=10)
    a = ap.parse_args()
    print(json.dumps(Ranker().rank(a.service, a.lat, a.lon, a.radius, a.w, a.top), indent=2, default=str))


Writing nb_src/rank_services.py


In [6]:
%%writefile nb_src/suggest.py
"""
"Suggested for your area": services ranked by LOCAL HEALTH NEED (NFHS-5 district indicators).

There is no search-log data, so we do not claim "most searched". Instead, for the district at the
user's location, each indicator is ranked against all ~700 districts; services linked to indicators
where the district is in the worst quarter nationally are suggested, each with a cited reason.
"""
import math
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree

# indicator column -> (plain-language label, higher_is_worse, services it points to)
INDICATORS = {
    "adult_high_sugar": ("adults with high blood sugar", True, ["diabetes_care", "dialysis", "diabetic_retinopathy_care"]),
    "adult_high_bp": ("adults with high blood pressure", True, ["electrocardiogram_ecg", "echocardiography"]),
    "all_w15_49_who_are_anaemic_pct": ("women aged 15-49 who are anaemic", True, ["blood_bank", "blood_transfusion"]),
    "births_delivered_by_csection_5y_pct": ("births delivered by C-section", True, ["c_section", "emergency_c_section"]),
    "institutional_birth_5y_pct": ("births in a health facility", False, ["vaginal_delivery", "antenatal_prenatal_care"]),
    "mothers_who_had_at_least_4_anc_visits_lb5y_pct": ("mothers with 4+ antenatal visits", False, ["antenatal_prenatal_care"]),
    "child_12_23m_fully_vaccinated_based_on_information_from_eit_pct": ("children aged 12-23 months fully vaccinated", False, ["vaccination_and_immunization"]),
    "women_age_30_49_years_ever_undergone_a_cervical_screen_pct": ("women aged 30-49 ever screened for cervical cancer", False, ["cervical_screening_pap_smear"]),
    "women_age_30_49_years_ever_undergone_a_breast_exam_pct": ("women aged 30-49 ever given a breast exam", False, ["mammography"]),
    "m15_plus_who_use_any_kind_of_tobacco_pct": ("men aged 15+ who use tobacco", True, ["cancer_screening"]),
    "fp_unmet_total_cm_w15_49_7_pct": ("women with unmet need for family planning", True, ["family_planning"]),
}
WORST_SHARE = 0.5           # suggest when the district is worse than the national median
MAX_KM = 75                 # farther than this from any district centre -> no suggestion


STATE_ALIASES = {"maharastra": "maharashtra", "nctofdelhi": "delhi"}   # NFHS spellings


def canon_state(s):
    s = re.sub(r"[^a-z]", "", str(s).lower()).replace("and", "")
    return STATE_ALIASES.get(s, s)


def key(state, district):
    return canon_state(state) + "|" + re.sub(r"[^a-z]", "", str(district).lower())


def num(v):
    """NFHS cells: '12.3', '(12.3)' = based on few cases, '*' = suppressed."""
    if isinstance(v, (int, float)):
        return (float(v), False) if math.isfinite(v) else (None, False)
    s = str(v).strip()
    if not s or s == "*":
        return None, False
    few = s.startswith("(")
    try:
        return float(s.strip("() ")), few
    except ValueError:
        return None, False


class Suggester:
    def __init__(self, health_csv, centroids_csv, valid_services):
        h = pd.read_csv(health_csv, low_memory=False)
        h["key"] = [key(s, d) for s, d in zip(h.state_ut, h.district_name)]
        h = h.drop_duplicates("key").set_index("key")
        cols = {}
        for c in [c for c in INDICATORS if c in h.columns] + [
                "w15_plus_with_high_or_very_high_gt_140_mg_dl_blood_sugar_or_pct", "m15_plus_with_high_or_very_high_gt_140_mg_dl_blood_sugar_or_pct",
                "w15_plus_with_high_bp_sys_gte_140_mmhg_and_or_dia_gte_90_mm_pct", "m15_plus_with_high_bp_sys_gte_140_mmhg_and_or_dia_gte_90_mm_pct"]:
            parsed = h[c].map(num)
            cols[c] = parsed.map(lambda t: t[0]); cols[c + "__few"] = parsed.map(lambda t: t[1])
        v = pd.DataFrame(cols, index=h.index)
        v["adult_high_sugar"] = v[["w15_plus_with_high_or_very_high_gt_140_mg_dl_blood_sugar_or_pct",
                                   "m15_plus_with_high_or_very_high_gt_140_mg_dl_blood_sugar_or_pct"]].mean(axis=1)
        v["adult_high_bp"] = v[["w15_plus_with_high_bp_sys_gte_140_mmhg_and_or_dia_gte_90_mm_pct",
                                "m15_plus_with_high_bp_sys_gte_140_mmhg_and_or_dia_gte_90_mm_pct"]].mean(axis=1)
        v["adult_high_sugar__few"] = v["adult_high_bp__few"] = False
        self.values = v
        # share of districts this district is worse than (0..1), per indicator
        self.worse_than = pd.DataFrame({c: (v[c].rank(pct=True) if hi else (-v[c]).rank(pct=True))
                                        for c, (_, hi, _) in INDICATORS.items()}, index=v.index)
        fix = {"Maharastra": "Maharashtra"}
        self.names = {k: f"{d.strip()}, {fix.get(s.strip(), s.strip())}" for k, d, s in zip(h.index, h.district_name, h.state_ut)}
        cen = pd.read_csv(centroids_csv)
        cen["key"] = [key(*k.split("|", 1)) for k in cen.key]
        cen = cen.groupby("key", as_index=False).agg(lat=("lat", "median"), lon=("lon", "median"))
        self.coverage = (int(cen.key.isin(self.values.index).sum()), len(cen))
        cen = cen[cen.key.isin(self.values.index)]
        self.cen_keys = cen.key.tolist()
        self.tree = BallTree(np.radians(cen[["lat", "lon"]].values), metric="haversine")
        self.valid = set(valid_services)

    def suggest(self, lat, lon, limit=5):
        d, i = self.tree.query(np.radians([[lat, lon]]), k=1)
        if d[0][0] * 6371 > MAX_KM:
            return []
        k = self.cen_keys[i[0][0]]
        out = {}
        for c, (label, hi, services) in INDICATORS.items():
            share, val = self.worse_than.at[k, c], self.values.at[k, c]
            if pd.isna(share) or pd.isna(val) or share < 1 - WORST_SHARE:
                continue
            few = bool(self.values.at[k, c + "__few"])
            reason = (f"{self.names[k]}: {val:.1f}% {label}, worse than {share * 100:.0f}% of districts "
                      f"(NFHS-5{', small sample' if few else ''}).")
            for s in services:
                if s in self.valid and (s not in out or share > out[s][0]):
                    out[s] = (share, reason)
        ranked = sorted(out.items(), key=lambda kv: -kv[1][0])[:limit]
        return [{"id": s, "reason": r} for s, (_, r) in ranked]


Writing nb_src/suggest.py


In [7]:
%%writefile nb_src/server.py
"""
CareFinder data backend: ranking + issue-report inbox.

  POST /rank      {"service_id", "origin": {"latitude", "longitude", "label"}, "radius_km"}
                  -> ranking JSON in the frontend's contract (+ fusion_method)
  POST /reports   report event from the frontend (Idempotency-Key header == report_id)
                  -> {"report_id": ..., "status": "pending"}   (201 new, 200 duplicate, 409 conflict)
  GET  /reports?status=pending&limit=100   pending reports for the admin review
  GET  /health

Run:  python care_backend/server.py      (port CARE_BACKEND_PORT, default 8001)
"""
import hmac
import json
import math
import threading
import os
import sqlite3
import sys
from contextlib import closing
from datetime import datetime, timezone
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import parse_qs, urlsplit
from uuid import UUID

HERE = Path(__file__).resolve().parent
DATA = Path(os.getenv("CARE_DATA_DIR", HERE / "data"))
os.environ.setdefault("DATA_FILE", str(DATA / "facilities_slim.csv") if (DATA / "facilities_slim.csv").exists()
                      else str(DATA / "facilities_final_9989.csv"))
os.environ.setdefault("DICT_FILE", str(DATA / "datathon-service-dictionary.yaml"))
os.environ.setdefault("OUT_DIR", str(DATA / "llm_out_all"))
sys.path.insert(0, str(HERE))

import rank_services  # noqa: E402

TOP_PER_LIST = int(os.getenv("CARE_TOP_PER_LIST", "25"))   # top-K of each of the three rankings
FUSION_TEXT = (
    "How we decide. (1) Eligibility: a facility is shown only if its own record states it provides this "
    "service. Each of ~10,000 records was read by an LLM and every claim is tied to a verbatim quote; "
    "referrals, planned services, other branches, equipment-only and passing mentions are excluded. "
    "Facilities outside your straight-line radius are excluded. "
    "(2) Best Match ranks by readiness: the share of the service's required staff, equipment and "
    "diagnostics that the record documents (WHO SARA method; requirements from our service dictionary, "
    "e.g. the national dialysis programme checklist for dialysis). Ties are broken by evidence certainty "
    "(GRADE-style: stated count or several fields > one field > specialty tag only; rated down for "
    "location conflicts or outdated records). "
    "(3) Nearest ranks by straight-line distance; facilities with suspect coordinates go last. "
    "(4) Balanced combines the Best Match and Nearest ranks with reciprocal rank fusion (k=60), weighting "
    "readiness and closeness equally. Trade-off: a closer facility with thinner documentation can outrank "
    "a better-documented one farther away; use Best Match when capability matters more than distance. "
    "Readiness reflects what the record documents, not an inspection: missing evidence is reported as "
    "unknown, never as absent.")
import yaml  # noqa: E402
AUTHORITY_TITLES = {k: v.get("title", k) for k, v in
                    (yaml.safe_load(open(os.environ["DICT_FILE"])).get("authorities") or {}).items()}
REPORTS_DB = Path(os.getenv("CARE_REPORTS_DB", DATA / "reports.sqlite3"))
ISSUES = {"service", "location", "facility_details", "evidence", "other"}
REPORT_KEYS = ("report_id", "facility_id", "service_id", "record_id", "issue_type",
               "suggested_change", "source_url", "data_mode", "status")

RANKER = None
RANKER_LOCK = threading.Lock()


def connect():
    REPORTS_DB.parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(REPORTS_DB, timeout=10)
    conn.row_factory = sqlite3.Row
    conn.execute("""CREATE TABLE IF NOT EXISTS facility_reports (
        report_id TEXT PRIMARY KEY, facility_id TEXT, service_id TEXT NOT NULL, record_id TEXT,
        issue_type TEXT NOT NULL, suggested_change TEXT NOT NULL, source_url TEXT,
        data_mode TEXT NOT NULL, status TEXT NOT NULL, received_at TEXT NOT NULL)""")
    cols = {r[1] for r in conn.execute("PRAGMA table_info(facility_reports)")}
    for col in ("decided_at", "decision"):
        if col not in cols:
            conn.execute(f"ALTER TABLE facility_reports ADD COLUMN {col} TEXT")
    conn.execute("""CREATE TABLE IF NOT EXISTS data_corrections (
        correction_id INTEGER PRIMARY KEY AUTOINCREMENT, report_id TEXT UNIQUE NOT NULL,
        facility_id TEXT, service_id TEXT, kind TEXT NOT NULL, change TEXT NOT NULL,
        old_value TEXT, applied_at TEXT NOT NULL)""")
    return conn


def load_corrections():
    with closing(connect()) as conn:
        rows = conn.execute("SELECT * FROM data_corrections ORDER BY correction_id").fetchall()
    return [dict(report_id=r["report_id"], facility_id=r["facility_id"], service_id=r["service_id"],
                 change=json.loads(r["change"])) for r in rows]


def rebuild_ranker():
    """Build a new ranker with all corrections applied, then swap it in (searches keep working)."""
    global RANKER
    with RANKER_LOCK:
        new = rank_services.Ranker(corrections=load_corrections())
        RANKER = new
    return new


print("Loading ranking data ...", flush=True)
rebuild_ranker()
from suggest import Suggester  # noqa: E402
SUGGESTER = Suggester(DATA / "district_health_nfhs5.csv", DATA / "district_centroids.csv", RANKER.S.keys())
print(f"Ranker ready: {len(RANKER.index)} services ({RANKER.build_seconds}s)", flush=True)


# ------------------------------ ranking ------------------------------
def clean(obj):
    """JSON-safe: NaN/inf -> None, numpy scalars -> Python."""
    if isinstance(obj, dict):
        return {k: clean(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [clean(v) for v in obj]
    if hasattr(obj, "item") and not isinstance(obj, (str, bytes)):
        obj = obj.item()
    if isinstance(obj, float) and not math.isfinite(obj):
        return None
    return obj


def search(payload):
    sid = payload["service_id"]
    lat, lon = float(payload["origin"]["latitude"]), float(payload["origin"]["longitude"])
    radius = payload["radius_km"]
    out = RANKER.rank(sid, lat, lon, float(radius), w=0.5, top=10 ** 9)
    n = out.get("n_candidates", 0)
    keep = [r for r in out["results"]
            if min(r["fused_rank"], r["distance_rank"], r["relevance_rank"]) <= TOP_PER_LIST]
    for r in keep:                                   # frontend rejects distance > radius
        r["distance_km"] = min(r["distance_km"], float(radius))
        explain(r, n)
    return clean(dict(service_id=sid, radius_km=radius, distance_method="straight_line",
                      results=keep, fusion_method=FUSION_TEXT,
                      n_candidates=out.get("n_candidates", 0)))


def explain(r, n):
    """Put WHY into the fields the UI displays: relevance_reason and uncertainties."""
    why = [r["relevance_reason"]]                                   # claim + evidence tier
    basis = r.get("confidence_basis", [])
    why.append(f"Evidence certainty: {r['confidence']}" + (f" ({'; '.join(basis)})." if basis else "."))
    docs, miss = r.get("items_documented", []), r.get("items_missing", [])
    if r.get("readiness") is None:
        why.append("Readiness: not assessed (this service has no staff/equipment requirement profile).")
    else:
        total = len(docs) + len(miss)
        why.append(f"Readiness {r['readiness']:.2f} (averaged over staff, equipment and diagnostics): "
                   f"{len(docs)} of {total} required items documented"
                   + (f" ({', '.join(docs)})" if docs else "") + ".")
    for a in r.get("authority_checks", []):
        why.append(f"{AUTHORITY_TITLES.get(a['ref'], a['ref'])}: {a['met']} of {a['total']} essential items documented.")
    why.append(f"Why this position: Balanced #{r['fused_rank']} of {n} combines Nearest #{r['distance_rank']} "
               f"({r['distance_km']:.1f} km) and Best Match #{r['relevance_rank']}.")
    r["relevance_reason"] = "\n".join(why)                         # one point per line in the UI
    if r["confidence"] in ("low", "very low"):
        r["uncertainties"] = [f"Evidence for this service is {r['confidence']} certainty; confirm with the "
                              "facility before travelling."] + r["uncertainties"]


# ------------------------------ reports ------------------------------
class BadReport(Exception):
    def __init__(self, status, message):
        super().__init__(message)
        self.status = status


def validate_report(body, key):
    if not isinstance(body, dict) or set(body) - set(REPORT_KEYS):
        raise BadReport(400, "Unexpected report fields.")
    try:
        rid = str(UUID(str(body.get("report_id"))))
    except ValueError:
        raise BadReport(400, "report_id must be a UUID.") from None
    if key != body.get("report_id"):
        raise BadReport(400, "Idempotency-Key header must equal report_id.")
    ev = {k: body.get(k) for k in REPORT_KEYS}
    ev["report_id"] = rid
    if ev["issue_type"] not in ISSUES:
        raise BadReport(400, "issue_type must be one of " + ", ".join(sorted(ISSUES)))
    if ev["data_mode"] not in ("demo", "live") or ev["status"] != "pending":
        raise BadReport(400, "data_mode must be demo|live and status must be pending.")
    for k, limit, required in (("service_id", 80, True), ("suggested_change", 2000, True),
                               ("facility_id", 256, False), ("record_id", 256, False),
                               ("source_url", 2048, False)):
        v = ev[k]
        if v is None and not required:
            continue
        if not isinstance(v, str) or not v.strip() or len(v) > limit:
            raise BadReport(400, f"Check the {k} field.")
    return ev


def store_report(ev):
    with closing(connect()) as conn, conn:
        cur = conn.execute(
            """INSERT INTO facility_reports (report_id, facility_id, service_id, record_id, issue_type,
               suggested_change, source_url, data_mode, status, received_at)
               VALUES (:report_id,:facility_id,:service_id,:record_id,
               :issue_type,:suggested_change,:source_url,:data_mode,:status,:received_at)
               ON CONFLICT(report_id) DO NOTHING""",
            {**ev, "received_at": datetime.now(timezone.utc).isoformat(timespec="seconds")})
        if cur.rowcount == 1:
            return 201
        saved = dict(conn.execute("SELECT * FROM facility_reports WHERE report_id=?", (ev["report_id"],)).fetchone())
        if any(saved[k] != ev[k] for k in REPORT_KEYS if k != "status"):
            raise BadReport(409, "That report_id already belongs to a different report.")
        return 200                                    # same report retried: idempotent


def list_reports(status, limit):
    with closing(connect()) as conn:
        q = "SELECT * FROM facility_reports" + (" WHERE status=?" if status else "") + \
            " ORDER BY received_at DESC LIMIT ?"
        rows = conn.execute(q, ((status, limit) if status else (limit,))).fetchall()
    out = []
    for r in rows:
        d = dict(r)
        d["change"] = json.loads(d.pop("decision") or "null")
        out.append(d)
    return out


# ------------------------------ admin ------------------------------
ADMIN_KEY = os.getenv("CARE_ADMIN_KEY", "")
DETAIL_FIELDS = {"name": 256, "address_line1": 256, "address_line2": 256, "address_city": 128,
                 "address_stateOrRegion": 128}


def with_names(rep):
    """Add human-readable facility and service names for the admin page."""
    f = RANKER.fac.get(rep.get("facility_id")) if rep.get("facility_id") else None
    rep["facility_name"] = f.get("name") if f else None
    rep["facility_address"] = ", ".join(str(f[k]) for k in ("address_line1", "address_city", "address_stateOrRegion")
                                        if f and isinstance(f.get(k), str)) or None
    svc = RANKER.S.get(rep.get("service_id")) or {}
    rep["service_name"] = svc.get("display_name", rep.get("service_id"))
    return clean(rep)


def admin_ok(header):
    if not ADMIN_KEY:
        raise BadReport(503, "Admin access is not configured (set CARE_ADMIN_KEY).")
    if not hmac.compare_digest((header or "").encode(), ("Bearer " + ADMIN_KEY).encode()):
        raise BadReport(401, "Wrong administrator key.")


def facility_snapshot(fid, sid):
    """Current values the admin sees before editing (from the live, corrected ranker)."""
    r = RANKER
    f = r.fac.get(fid)
    if f is None:
        raise BadReport(404, "Unknown facility.")
    c = r.claims[(r.claims.facility_id == fid) & (r.claims.service_id == sid)] if sid else r.claims.iloc[0:0]
    offered = bool(len(c) and c.gate_pass.astype(bool).any())
    quotes = [dict(field=e.field, quote=e.quote) for e in r.ev_idx.get((fid, f"{sid}_offered"), [])[:3]]
    modality = c.modality.iloc[0] if len(c) and isinstance(c.modality.iloc[0], str) else None
    resources = []
    if sid and sid in r.S:
        slots, _ = r.profile(sid, modality)
        for res in dict.fromkeys(x for alts in slots for x in alts):
            row = r.slot_idx.get((fid, res))
            st = getattr(row, "status", None)
            resources.append(dict(resource=res, label=r.name_of(res), kind=r.R[res]["kind"],
                                  status={"confirmed": "present", "contradicted": "absent"}.get(st, "unknown"),
                                  count=getattr(row, "count", None),
                                  quotes=[dict(field=e.field, quote=e.quote) for e in r.ev_idx.get((fid, res), [])[:6]]))
    return clean(dict(facility_id=fid, service_id=sid,
                      **{k: f.get(k) for k in DETAIL_FIELDS}, latitude=f["lat"], longitude=f["lon"],
                      location_flagged=bool(f["geo_flag"]), location_note=f.get("location_note"),
                      service_offered=offered, service_claim=(c.claim_type.iloc[0] if len(c) else None),
                      service_quotes=quotes, resources=resources))


def validate_change(report, change):
    if not isinstance(change, dict):
        raise BadReport(400, "Accepting a report needs an explicit data change.")
    kind = change.get("kind")
    fid = report["facility_id"]
    if kind == "none":
        return {"kind": "none", "note": str(change.get("note", ""))[:1000]}
    if not fid:
        raise BadReport(400, "This report is not about a facility; only 'no data change' is possible.")
    if kind == "edit":
        ops, out = change.get("ops"), []
        note = change.get("note")
        if not isinstance(ops, list) or not ops:
            raise BadReport(400, "Change at least one value, or accept without a data change.")
        if not isinstance(note, str) or len(note.strip()) < 5:
            raise BadReport(400, "Say how the change was verified (at least 5 characters).")
        for op in ops:
            k = op.get("op") if isinstance(op, dict) else None
            if k == "details":
                out.append({"op": k, **validate_change(report, {"kind": "set_details", "fields": op.get("fields")})}); out[-1].pop("kind")
            elif k == "location":
                v = validate_change(report, {"kind": "set_location", "latitude": op.get("latitude"), "longitude": op.get("longitude")})
                out.append({"op": k, "latitude": v["latitude"], "longitude": v["longitude"]})
            elif k == "service" and isinstance(op.get("offered"), bool):
                out.append({"op": k, "offered": op["offered"]})
            elif k == "resource" and op.get("status") in ("present", "absent", "unknown") and isinstance(op.get("resource"), str):
                cnt = op.get("count")
                if cnt is not None and not (isinstance(cnt, int) and 0 <= cnt <= 100000):
                    raise BadReport(400, "Counts must be whole numbers.")
                out.append({"op": k, "resource": op["resource"], "status": op["status"], "count": cnt})
            elif k == "remove_quote" and isinstance(op.get("slot"), str) and isinstance(op.get("quote"), str):
                out.append({"op": k, "slot": op["slot"], "quote": op["quote"]})
            else:
                raise BadReport(400, "Unrecognised edit.")
        return {"kind": "edit", "ops": out, "note": note.strip()[:1000]}
    if kind == "set_location":
        lat, lon = change.get("latitude"), change.get("longitude")
        if not all(isinstance(v, (int, float)) and math.isfinite(v) for v in (lat, lon)) \
                or not (6 <= lat <= 37.5 and 68 <= lon <= 97.5):
            raise BadReport(400, "Enter coordinates inside India.")
        return {"kind": kind, "latitude": float(lat), "longitude": float(lon)}
    if kind == "set_details":
        fields = change.get("fields")
        if not isinstance(fields, dict) or not fields or set(fields) - set(DETAIL_FIELDS):
            raise BadReport(400, "Choose facility details to change.")
        for k, v in fields.items():
            if not isinstance(v, str) or not v.strip() or len(v) > DETAIL_FIELDS[k]:
                raise BadReport(400, f"Check the {k} value.")
        return {"kind": kind, "fields": {k: v.strip() for k, v in fields.items()}}
    if kind == "set_service":
        note = change.get("note")
        if not isinstance(change.get("offered"), bool) or not isinstance(note, str) or len(note.strip()) < 5:
            raise BadReport(400, "Say whether the service is offered and how it was verified (min. 5 characters).")
        return {"kind": kind, "offered": change["offered"], "note": note.strip()[:1000]}
    raise BadReport(400, "Unknown change kind.")


def decide(body):
    if not isinstance(body, dict) or body.get("status") not in ("accepted", "rejected"):
        raise BadReport(400, "Send report_id and status accepted|rejected.")
    rid, status = str(body.get("report_id")), body["status"]
    with closing(connect()) as conn, conn:
        rep = conn.execute("SELECT * FROM facility_reports WHERE report_id=?", (rid,)).fetchone()
        if rep is None:
            raise BadReport(404, "Unknown report.")
        rep = dict(rep)
        change = validate_change(rep, body.get("change")) if status == "accepted" else None
        if rep["status"] != "pending":                       # idempotent retry vs conflicting decision
            if rep["status"] == status and json.loads(rep["decision"] or "null") == change:
                return {"report_id": rid, "status": status, "change": change, "data_updated": False}
            raise BadReport(409, f"This report was already {rep['status']}.")
        now = datetime.now(timezone.utc).isoformat(timespec="seconds")
        if change and change["kind"] != "none":
            old = facility_snapshot(rep["facility_id"], rep["service_id"])
            conn.execute("""INSERT INTO data_corrections (report_id, facility_id, service_id, kind, change, old_value, applied_at)
                            VALUES (?,?,?,?,?,?,?)""", (rid, rep["facility_id"], rep["service_id"], change["kind"],
                                                        json.dumps(change), json.dumps(old), now))
        conn.execute("UPDATE facility_reports SET status=?, decided_at=?, decision=? WHERE report_id=?",
                     (status, now, json.dumps(change) if change else None, rid))
    updated = bool(change and change["kind"] != "none")
    if updated:
        rebuild_ranker()                                      # search results reflect the change now
    return {"report_id": rid, "status": status, "change": change, "data_updated": updated}


# ------------------------------ HTTP ------------------------------
class Handler(BaseHTTPRequestHandler):
    def log_message(self, fmt, *args):
        pass

    def reply(self, code, obj):
        data = json.dumps(obj, allow_nan=False).encode()
        self.send_response(code)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        url = urlsplit(self.path)
        if url.path == "/health":
            return self.reply(200, {"ok": True, "services": len(RANKER.index)})  # noqa
        if url.path == "/reports":
            q = parse_qs(url.query)
            return self.reply(200, list_reports(q.get("status", ["pending"])[0],
                                                min(int(q.get("limit", ["100"])[0]), 1000)))
        if url.path in ("/admin/reports", "/admin/facility", "/admin/corrections"):
            try:
                admin_ok(self.headers.get("Authorization"))
                q = parse_qs(url.query)
                if url.path == "/admin/reports":
                    return self.reply(200, [with_names(r) for r in list_reports(None, 1000)])
                if url.path == "/admin/facility":
                    return self.reply(200, facility_snapshot(q.get("facility_id", [""])[0],
                                                             q.get("service_id", [None])[0]))
                with closing(connect()) as conn:
                    return self.reply(200, [dict(r) for r in conn.execute(
                        "SELECT * FROM data_corrections ORDER BY correction_id DESC")])
            except BadReport as e:
                return self.reply(e.status, {"error": str(e)})
        self.reply(404, {"error": "not found"})

    def do_POST(self):
        path = urlsplit(self.path).path
        try:
            size = int(self.headers.get("Content-Length", "0"))
            body = json.loads(self.rfile.read(size)) if 0 < size <= 65536 else None
        except (ValueError, UnicodeError):
            body = None
        if path == "/rank":
            try:
                return self.reply(200, search(body))
            except (KeyError, TypeError, ValueError):
                return self.reply(400, {"error": "Check service_id, origin and radius_km."})
        if path == "/suggestions":
            try:
                loc = body["location"]
                return self.reply(200, SUGGESTER.suggest(float(loc["latitude"]), float(loc["longitude"])))
            except (KeyError, TypeError, ValueError):
                return self.reply(400, {"error": "Send {location: {latitude, longitude}}."})
        if path == "/admin/reports":
            try:
                admin_ok(self.headers.get("Authorization"))
                return self.reply(200, decide(body))
            except BadReport as e:
                return self.reply(e.status, {"error": str(e)})
        if path == "/reports":
            try:
                ev = validate_report(body, self.headers.get("Idempotency-Key"))
                code = store_report(ev)
                return self.reply(code, {"report_id": ev["report_id"], "status": "pending"})
            except BadReport as e:
                return self.reply(e.status, {"error": str(e)})
        self.reply(404, {"error": "not found"})


if __name__ == "__main__":
    port = int(os.getenv("CARE_BACKEND_PORT", "8001"))
    print(f"CareFinder backend on 127.0.0.1:{port}", flush=True)
    ThreadingHTTPServer(("127.0.0.1", port), Handler).serve_forever()


Writing nb_src/server.py


## Part 1: Exploration
Findings that shaped the design:
* **List columns are stored as NumPy-array text** (e.g. `['CT scanner' 'MRI scanner']`, with no commas), so a JSON parser silently reads every list as empty. We parse the format explicitly.
* **Most evidence is free text.** Bed and doctor counts exist for only a minority of facilities, so readiness must come from text evidence.
* **Specialty tags are inflated.** Rows carry many repeated tags, so a tag alone is treated as the weakest evidence tier.
* **Locations can contradict addresses.** A "Nagpur" hospital has Pune coordinates. A city-level consistency check, combined with the upstream postcode-distance flag, marks such facilities, lowers their certainty, and ranks them last by distance.

In [8]:
os.environ["DATA_FILE"] = str(FULL) if FULL else os.environ["DATA_FILE"]
import llm_preprocess as L
df = (pd.read_parquet(FULL) if str(FULL).endswith(".parquet") else pd.read_csv(FULL, low_memory=False)) if FULL else None
if df is not None:
    print(f"facilities: {len(df):,}  unique ids: {df.unique_id.nunique():,}")
    raw = df["equipment"].dropna().astype(str).iloc[0]
    print("raw list cell:", raw[:80], "...")
    print("  json.loads ->", end=" ");
    try: json.loads(raw); print("ok")
    except Exception as e: print("fails:", type(e).__name__, "| our parser ->", L.as_list(raw)[:3])
    rows = []
    for c in ["procedure", "equipment", "capability", "specialties"]:
        lists = df[c].map(L.as_list)
        rows.append(dict(field=c, rows_with_items=f"{(lists.map(len) > 0).mean():.0%}", median_items=int(lists.map(len).median()),
                         median_unique=int(lists.map(lambda l: len(set(l))).median())))
    for c in ["numberDoctors", "capacity"]:
        rows.append(dict(field=c, rows_with_items=f"{pd.to_numeric(df[c], errors='coerce').notna().mean():.0%}"))
    display(pd.DataFrame(rows))
else:
    print("Put facilities_final_9989.parquet (or .csv) in", DATA_DIR, "to run Part 1-2 profiling.")

facilities: 9,989  unique ids: 9,989
raw list cell: ['CT scanner' 'MRI scanner' 'Dialysis machines: 10'
 '50 MT LMO storage tank' '2 ...
  json.loads -> fails: JSONDecodeError | our parser -> ['CT scanner', 'MRI scanner', 'Dialysis machines: 10']


,field,rows_with_items,median_items,median_unique
0,procedure,89%,10.0,10.0
1,equipment,71%,2.0,2.0
2,capability,100%,18.0,18.0
3,specialties,100%,10.0,10.0
4,numberDoctors,36%,NaN,NaN
5,capacity,25%,NaN,NaN


In [9]:
import rank_services as RS
ranker = RS.Ranker()
flagged = sum(v["geo_flag"] for v in ranker.fac.values())
print(f"facilities with suspect coordinates (city/state consistency + upstream postcode flag): {flagged:,}")
nag = [v for v in ranker.fac.values() if "Babasaheb Ambedkar Hospital & Research Center, Nagpur" in str(v["name"])]
if nag: print("example:", nag[0]["name"], "| coordinates:", round(nag[0]["lat"], 3), round(nag[0]["lon"], 3), "(Pune) | flagged:", nag[0]["geo_flag"])

facilities with suspect coordinates (city/state consistency + upstream postcode flag): 2,365
example: Dr. Babasaheb Ambedkar Hospital & Research Center, Nagpur | coordinates: 18.503 73.877 (Pune) | flagged: True


## Part 2: Evidence extraction
Each record is split into numbered items (P1… procedures, E1… equipment, C1… capabilities, D1… description sentences, S1… specialty tags). The LLM (DeepSeek, JSON mode) receives only the candidate services found by a lenient keyword pre-filter and their required resources, and must **cite item IDs**. Quotes are looked up in code, so **every quote shown to users is verbatim**.

Each mention is classified as offered, listed, negated, planned, referral-only, other-branch, equipment-only, specialty-only, or a passing mention. Text like "patients needing dialysis are referred elsewhere" or "Upcoming Dialysis Wing" is therefore never counted as the service.

Confidence is computed from the evidence, never by the LLM: a stated count, or two or more fields, ranks above one field, which ranks above a specialty tag only.

In [10]:
claims = pd.read_csv(DATA_DIR / "llm_out_all/service_claims.csv")
slots = pd.read_csv(DATA_DIR / "llm_out_all/slot_status.csv")
ev = pd.read_csv(DATA_DIR / "llm_out_all/evidence.csv", low_memory=False)
print(f"facility-service decisions: {len(claims):,} | confirmed: {claims.gate_pass.sum():,} | services covered: {claims[claims.gate_pass].service_id.nunique()}")
print("decision source:", claims.method.value_counts().to_dict())
display(claims.claim_type.value_counts().rename("count").to_frame())

facility-service decisions: 88,223 | confirmed: 80,157 | services covered: 186
decision source: {'llm': 88210, 'rule': 13}


,count
claim_type,
explicitly_offered_at_branch,64462
listed_on_branch_service_menu,15695
equipment_only,3163
specialty_only,2306
mention_without_service_claim,1982
other_branch,361
planned_only,95
negated,82
referral_only,77


In [11]:
# Verbatim check: every quote must be an exact substring of the field it cites
if df is not None:
    idx = df.set_index("unique_id")
    sample = ev[ev.field.isin(["procedure","equipment","capability","description","specialties"])].sample(3000, random_state=0)
    def present(r):
        cell = idx.at[r.facility_id, r.field]
        items = L.as_list(cell) if r.field != "description" else [cell]
        return any(str(r.quote) in str(x) for x in items)
    ok = sample.apply(present, axis=1)
    print(f"verbatim quotes in a random sample of 3,000 evidence rows: {ok.mean():.2%}")

verbatim quotes in a random sample of 3,000 evidence rows: 100.00%


In [12]:
# LLM vs the deterministic rule baseline (same facilities, all 186 services)
if df is not None:
    import io, contextlib
    os.environ.update(OUT_DIR="nb_rules_out", USE_LLM="0", SERVICES="all")
    import importlib; importlib.reload(L)
    with contextlib.redirect_stdout(io.StringIO()): L.main()
    rul = pd.read_csv("nb_rules_out/service_claims.csv"); rsl = pd.read_csv("nb_rules_out/slot_status.csv")
    m = claims.merge(rul, on=["facility_id", "service_id"], how="outer", suffixes=("_llm", "_rule"))
    gl, gr = m.gate_pass_llm.fillna(False).astype(bool), m.gate_pass_rule.fillna(False).astype(bool)
    display(pd.DataFrame([
        ["service decisions compared", f"{len(m):,}", ""],
        ["agreement on 'offers the service'", f"{(gl == gr).mean():.1%}", ""],
        ["confirmed facility-service pairs", f"{gr.sum():,}", f"{gl.sum():,}"],
        ["confirmed supporting resources", f"{(rsl.status == 'confirmed').sum():,}", f"{(slots.status == 'confirmed').sum():,}"],
        ["stated counts extracted", f"{rsl['count'].notna().sum():,}", f"{slots['count'].notna().sum():,}"],
        ["rejected mentions (negated/referral/planned/other branch)",
         f"{rul.claim_type.isin(['negated','referral_only','planned_only','other_branch']).sum():,}",
         f"{claims.claim_type.isin(['negated','referral_only','planned_only','other_branch']).sum():,}"],
    ], columns=["metric", "rules", "LLM (final)"]).set_index("metric"))
    os.environ["OUT_DIR"] = str(DATA_DIR / "llm_out_all")

,rules,LLM (final)
metric,,
service decisions compared,"89,056",
agreement on 'offers the service',90.3%,
confirmed facility-service pairs,"82,938","80,157"
confirmed supporting resources,"4,096","23,498"
stated counts extracted,160,929
rejected mentions (negated/referral/planned/other branch),163,615


**Adjudication.** A second, focused LLM pass re-asked each case where rules matched a service but the first pass said "none", one facility and one service at a time, using only the matched lines. Of 3,151 such pairs, **851 were recovered** (e.g. "Has a Blood Bank", "Intensive Care Unit (ICU)"). The rest were confirmed as rule false positives: "Pediatrics" and "General Surgery" listed separately, a company name containing "ICU", a PubMed affiliation.

## Part 3: Scoring and ranking
| Step | Method | Source |
|---|---|---|
| Eligibility | All essential evidence must be confirmed, all-or-nothing | WHO EmONC signal-function classification |
| Readiness | Required items present or absent, averaged within staff / equipment / diagnostics, then across domains | WHO SARA service-specific readiness |
| Requirements | Per-service items, e.g. PMNDP haemodialysis checklist; IPHS 2022 | NHSRC / MoHFW |
| Certainty | Four levels (high / moderate / low / very low), lowered one level per data-quality concern | GRADE; WHO Data Quality Review |
| Balanced rank | Reciprocal rank fusion of the Nearest and Best Match ranks, k = 60 | Cormack, Clarke & Büttcher, SIGIR 2009 |

Facilities with suspect coordinates are ranked last on Nearest, because their distance is unreliable. Missing evidence is reported as **unknown**, never as absent.

In [13]:
import server     # the exact backend the web app calls (loads ranker + suggester)
print("HOW (shown in the app's method panel):\n"); print(server.FUSION_TEXT)

Loading ranking data ...


Ranker ready: 186 services (4.69s)


HOW (shown in the app's method panel):

How we decide. (1) Eligibility: a facility is shown only if its own record states it provides this service. Each of ~10,000 records was read by an LLM and every claim is tied to a verbatim quote; referrals, planned services, other branches, equipment-only and passing mentions are excluded. Facilities outside your straight-line radius are excluded. (2) Best Match ranks by readiness: the share of the service's required staff, equipment and diagnostics that the record documents (WHO SARA method; requirements from our service dictionary, e.g. the national dialysis programme checklist for dialysis). Ties are broken by evidence certainty (GRADE-style: stated count or several fields > one field > specialty tag only; rated down for location conflicts or outdated records). (3) Nearest ranks by straight-line distance; facilities with suspect coordinates go last. (4) Balanced combines the Best Match and Nearest ranks with reciprocal rank fusion (k=60), weig

In [14]:
def show(service, lat, lon, label, radius, top=3):
    out = server.search({"service_id": service, "origin": {"latitude": lat, "longitude": lon, "label": label}, "radius_km": radius})
    print(f"=== {service} near {label}, {radius} km: {out['n_candidates']} eligible facilities ===")
    for r in sorted(out["results"], key=lambda r: r["fused_rank"])[:top]:
        print(f"\n#{r['fused_rank']} {r['facility_name']} ({r['distance_km']} km) | confidence: {r['confidence']}")
        print("  WHY:\n   - " + r["relevance_reason"].replace("\n", "\n   - "))
        for e in r["evidence"]: print(f"  EVIDENCE [{e['field']}]: \"{e['quote']}\"")
        print("  UNCERTAIN:", r["uncertainties"][:3])
show("dialysis", 18.5204, 73.8567, "Pune", 15)

=== dialysis near Pune, 15 km: 21 eligible facilities ===

#1 Galaxy Care Hospital, Pune (2.31 km) | confidence: moderate
  WHY:
   - Dialysis (any modality) is explicitly offered at this facility (in one field). Supporting evidence: dialysis machine, icu.
   - Evidence certainty: moderate (evidence in one field).
   - Readiness 0.33 (averaged over staff, equipment and diagnostics): 2 of 7 required items documented (dialysis machine, icu).
   - PMNDP self-assessment checklist for haemodialysis services: 1 of 2 essential items documented.
   - Why this position: Balanced #1 of 21 combines Nearest #3 (2.3 km) and Best Match #6.
  EVIDENCE [procedure]: "Dialysis"
  EVIDENCE [equipment]: "On-site dialysis equipment"
  EVIDENCE [capability]: "ICU services"
  UNCERTAIN: ['No evidence of dialysis ro plant.', 'No evidence of dialysis nurse or dialysis technician.', 'No evidence of nephrologist.']

#2 Aditya Birla Memorial Hospital (9.94 km) | confidence: high
  WHY:
   - Dialysis (any modality

In [15]:
show("cataract_surgery", 28.6139, 77.2090, "Delhi", 15, top=2)

=== cataract_surgery near Delhi, 15 km: 28 eligible facilities ===

#1 Khanna Medicare Centre (6.47 km) | confidence: high
  WHY:
   - Cataract surgery is listed in this facility's services (in multiple fields).
   - Evidence certainty: high (evidence in multiple fields).
   - Readiness: not assessed (this service has no staff/equipment requirement profile).
   - Why this position: Balanced #1 of 28 combines Nearest #4 (6.5 km) and Best Match #1.
  EVIDENCE [procedure]: "Stitch-less cataract surgery"
  UNCERTAIN: ['Readiness not assessed: the service profile has no staff/equipment requirements.', 'No update date available for this record.']

#2 Amvision Eye & Child Care (7.78 km) | confidence: high
  WHY:
   - Cataract surgery is explicitly offered at this facility (in multiple fields).
   - Evidence certainty: high (evidence in multiple fields).
   - Readiness: not assessed (this service has no staff/equipment requirement profile).
   - Why this position: Balanced #2 of 28 combines Ne

## Part 4: Online query engine
Everything that does not depend on the user's location is precomputed per service at startup: eligibility, readiness, certainty, checklist results and explanation inputs. A **per-service BallTree with the haversine metric** answers radius queries in O(log n + k); evidence and facility lookups are hash maps; and ranking is a NumPy `lexsort` over only the k facilities in range. Each response returns the top 25 of all three rankings, so every tab in the UI stays correct.

In [16]:
cities = {"Pune": (18.5204, 73.8567), "Mumbai": (19.0760, 72.8777), "Delhi": (28.6139, 77.2090), "Bengaluru": (12.9716, 77.5946),
          "Chennai": (13.0827, 80.2707), "Kolkata": (22.5726, 88.3639), "Jaipur": (26.9124, 75.7873), "Patna": (25.5941, 85.1376)}
R = server.RANKER; random.seed(1); svcs = list(R.index); ts = []
for _ in range(500):
    s = random.choice(svcs); la, lo = random.choice(list(cities.values()))
    t = time.perf_counter(); R.rank(s, la, lo, random.choice([10, 30, 50, 4000]), 0.5, 10); ts.append((time.perf_counter() - t) * 1000)
print(f"services indexed: {len(R.index)} | startup {R.build_seconds}s | 500 random queries (incl. all-India): median {statistics.median(ts):.2f} ms, p95 {sorted(ts)[475]:.2f} ms")

services indexed: 186 | startup 4.69s | 500 random queries (incl. all-India): median 0.27 ms, p95 0.51 ms


## Part 5: Local need (NFHS-5), suggestions and impact
The service menu suggests services by **local health need**, not search popularity (no search logs exist). The user's location maps to a district (centres computed from facilities linked to NFHS-5 districts). Each NFHS-5 indicator is ranked against all ~700 districts, and services tied to indicators where the district is worse than the national median are suggested, each with its reason.

In [17]:
for city in ["Delhi", "Mumbai", "Patna"]:
    print(city); [print("  ", s["id"], "|", s["reason"]) for s in server.SUGGESTER.suggest(*cities[city])[:3]]

Delhi
   electrocardiogram_ecg | Central, NCT of Delhi: 34.0% adults with high blood pressure, worse than 96% of districts (NFHS-5).
   echocardiography | Central, NCT of Delhi: 34.0% adults with high blood pressure, worse than 96% of districts (NFHS-5).
   c_section | Central, NCT of Delhi: 25.3% births delivered by C-section, worse than 67% of districts (NFHS-5).
Mumbai
   diabetes_care | Mumbai Suburban, Maharashtra: 20.4% adults with high blood sugar, worse than 91% of districts (NFHS-5).
   dialysis | Mumbai Suburban, Maharashtra: 20.4% adults with high blood sugar, worse than 91% of districts (NFHS-5).
   diabetic_retinopathy_care | Mumbai Suburban, Maharashtra: 20.4% adults with high blood sugar, worse than 91% of districts (NFHS-5).
Patna
   antenatal_prenatal_care | Patna, Bihar: 17.9% mothers with 4+ antenatal visits, worse than 98% of districts (NFHS-5).
   vaccination_and_immunization | Patna, Bihar: 56.6% children aged 12-23 months fully vaccinated, worse than 94% of distr

In [18]:
# Impact: districts with high need but no confirmed facility for the matching service in this dataset
import suggest as SG
sg = server.SUGGESTER
if df is not None:
    f = df[df.health_indicator_match_status.eq("exact_match")].copy()
    f["k"] = [SG.key(*k.split("|", 1)) for k in f.health_indicator_join_key]
    conf = claims[claims.gate_pass]; covered = set(f.k); rows = []
    for ind, svc, label in [("adult_high_sugar", ["dialysis"], "High blood sugar → dialysis"),
                            ("adult_high_bp", ["electrocardiogram_ecg", "echocardiography"], "High blood pressure → ECG / echo"),
                            ("all_w15_49_who_are_anaemic_pct", ["blood_bank", "blood_transfusion"], "Anaemia (women) → blood bank"),
                            ("women_age_30_49_years_ever_undergone_a_cervical_screen_pct", ["cervical_screening_pap_smear"], "Low cervical screening → Pap smear")]:
        worst = sg.worse_than[ind]; bad = set(worst[worst >= 0.75].index) & covered
        have = set(f[f.unique_id.isin(conf[conf.service_id.isin(svc)].facility_id)].k)
        miss = sorted(bad - have, key=lambda k: -sg.values.at[k, ind])
        rows.append([label, len(bad), len(miss), "; ".join(f"{sg.names[k]} ({sg.values.at[k, ind]:.1f}%)" for k in miss[:3])])
    display(pd.DataFrame(rows, columns=["need (district in worst quarter)", "districts", "no confirmed facility in data", "examples"]).set_index("need (district in worst quarter)"))
    print("NFHS-5 districts with no facility in the dataset at all:", len(set(sg.values.index) - covered), "of", len(sg.values))

,districts,no confirmed facility in data,examples
need (district in worst quarter),,,
High blood sugar → dialysis,126,52,"Mahe, Puducherry (27.9%); Prakasam, Andhra Pra..."
High blood pressure → ECG / echo,120,57,"South District, Sikkim (45.3%); Pathanamthitta..."
Anaemia (women) → blood bank,93,65,"Kishtwar, Jammu & Kashmir (84.8%); Udalguri, A..."
Low cervical screening → Pap smear,131,111,"Gwalior, Madhya Pradesh (0.2%); Anantnag, Jamm..."


NFHS-5 districts with no facility in the dataset at all: 268 of 706


**Policy proposals**
1. **Target dialysis-programme (PMNDP) expansion, cardiovascular screening and blood-bank capacity** at the need-without-supply districts above.
2. **Mandate structured facility service registries.** Most readiness evidence exists only as free text, and is often missing.
3. **Make crowd verification part of data maintenance:** user reports become admin-verified, audited corrections (Part 6).

*Caveat:* "no confirmed facility **in this dataset**" does not mean none exists. The dataset is largely private facilities, and many districts have no facilities in it at all.

## Part 6: Feedback loop
Users report problems from a facility's evidence view, choosing the exact quoted statement they dispute. An administrator reviews each report and must make an **explicit data edit**: facility details, location, service status, a requirement's status or count, or removing a quoted statement. Every edit carries a verification note and is stored with its old value in an audit table, then applied over the base tables; results cite it as "verified by an administrator". Below, the same mechanism is applied in memory to a real report ("Has 2 ICU beds": no beds).

In [19]:
fid = next(k for k, v in server.RANKER.fac.items() if v["name"] == "Soni Hospital")
edit = dict(facility_id=fid, service_id="dialysis", report_id="demo-report",
            change={"kind": "edit", "note": "Called the hospital: no ICU beds.",
                    "ops": [{"op": "resource", "resource": "icu", "status": "absent", "count": None},
                            {"op": "remove_quote", "slot": "icu", "quote": "Has 2 ICU beds"}]})
corrected = RS.Ranker(corrections=[edit])
for name, r in [("before", server.RANKER), ("after ", corrected)]:
    row = r.slot_idx.get((fid, "icu")); quotes = [e.quote for e in r.ev_idx.get((fid, "icu"), [])]
    print(f"{name}: ICU status = {getattr(row, 'status', 'unknown')}, tier = {getattr(row, 'tier', None)}, quotes = {quotes[:2]}")

before: ICU status = confirmed, tier = explicit_count, quotes = ['Has 2 ICU beds', 'ISCMM Level-III ICU certification for S. K. Soni Hospital']
after : ICU status = contradicted, tier = admin_verified, quotes = ['ISCMM Level-III ICU certification for S. K. Soni Hospital', 'Dedicated Neuro-ICU within Neurosciences Centre']


## Limitations
* Readiness reflects what records **document**, not inspections.
* The keyword pre-filter bounds recall: a service described in terms absent from the dictionary is never sent to the LLM.
* Distances are straight-line, not travel time.
* Reports and corrections are stored in SQLite, which is not durable on Databricks Apps; production would use a Unity Catalog volume.

## References
1. WHO SARA, service-specific readiness score: https://www.measureevaluation.org/rbf/indicator-collections/structural-indicators/specific-services-readiness-score
2. Availability of functional EmONC facilities (WHO signal functions): https://data4impactproject.org/?p=5667
3. GRADE certainty of evidence: https://bestpractice.bmj.com/info/evidence/learn-ebm/what-is-grade/
4. WHO Data Quality Assurance, Module 2 (desk review): https://www.who.int/publications/i/item/9789240047389
5. Cormack, Clarke, Büttcher (2009). Reciprocal Rank Fusion. SIGIR: https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf
6. NHSRC, PMNDP self-assessment checklist for haemodialysis services: https://www.nhsrcindia.org/sites/default/files/SELF%20ASSESSMENT%20CHECKLIST%20FOR%20HAEMODIALYSIS%20SERVICES-PMNDP.pdf
7. MoHFW, Indian Public Health Standards 2022, Vol. I: https://nhsrcindia.org/sites/default/files/Volume%201_SDH-DH.pdf
8. IIPS & MoHFW, NFHS-5 (2019–21): https://microdata.worldbank.org/index.php/catalog/4482/study-description
9. scikit-learn BallTree: https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.BallTree.html
10. Databricks Apps compute sizes: https://docs.databricks.com/en/dev-tools/databricks-apps/compute-size.html